<p class="lead">A virtual screen is, from the data scientist's point of view, a sorting problem with an expensive verification step. A model orders a library of molecules; the laboratory can afford to buy and test a few dozen of them; the question is whether the top of that order contains more active compounds than a random draw would, and by how much. This unit never runs a docking program and never opens a protein structure: it starts where the group's screening pipelines end, with a table of molecules and scores, and teaches the analytics of a ranked list — the metrics that measure early recognition, the confidence intervals they need, the ways a benchmark can flatter a method that has learnt nothing about pharmacology, and the arithmetic that converts an impressive enrichment factor into the sober number of compounds a student will actually have to order.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata el análisis estadístico del resultado de un cribado virtual: una lista ordenada de moléculas. Se construyen cuatro rankings basados en ligando (máxima similitud de Tanimoto a activos conocidos, un puntaje de cinco vecinos más cercanos sobre huellas moleculares, la probabilidad de un modelo QSAR reajustado aquí con <code>ranger</code> y una distancia de Mahalanobis en el espacio de descriptores) y se evalúan desde los primeros principios con la curva ROC y su área, los factores de enriquecimiento al 1 % y 5 %, BEDROC y RIE con su parámetro alfa, y la tasa de aciertos entre los primeros k, con intervalos de confianza obtenidos por bootstrap estratificado sobre activos y señuelos. Luego se desmonta el propio banco de pruebas: señuelos emparejados por propiedades frente a señuelos aleatorios, la sensibilidad al conjunto de referencia, el diagnóstico de enriquecimiento artificial (un bosque aleatorio que separa activos de señuelos usando solo propiedades fisicoquímicas) y el efecto de una prevalencia realista de 1 % y 0,1 %. Finalmente se comparan cuatro esquemas de consenso con el mejor ranking individual mediante un bootstrap pareado, se examina la correlación entre puntaje y tamaño molecular junto con la eficiencia de ligando, y se selecciona una lista de cincuenta compuestos con un criterio de diversidad defendible.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Evaluate a screening campaign with the right metrics — ROC-AUC, enrichment factor at 1 % and 5 %, BEDROC, RIE and hit rate at k — implementing each from its definition in R and attaching bootstrap confidence intervals.</li>
<li>Explain decoy design (property matching, the DUD-E logic), analogue bias and artificial enrichment, and why a good area under the curve on a convenient decoy set can mean very little.</li>
<li>Combine several ranking criteria (z-score, rank-by-rank, rank-by-vote, exponential consensus) and test with a paired bootstrap whether the consensus actually improves on the best single ranking.</li>
<li>Analyse a screening output table: score distributions, score-property correlations and size bias, the diversity of a hit list, and the selection of compounds for testing under a fixed budget.</li>
<li>Translate a retrospective enrichment into a prospective expectation using the base rate of actives in a real library.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running; every number quoted in the prose is printed by a chunk of this same document. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","Matrix","ranger","pROC","patchwork"))</code> once before the first chunk. The unit assumes the validation vocabulary of <a href="../U12_Validation/index.html">Unit 12</a>, the fingerprints and descriptors of <a href="../U13_Features_Dimensionality/index.html">Unit 13</a>, the ensembles of <a href="../U14_Ensembles_Tuning/index.html">Unit 14</a>, the clustering of <a href="../U15_Clustering/index.html">Unit 15</a> and the base-rate reasoning of <a href="../U10_Bayesian/index.html">Unit 10</a>. The whole document knits in about two minutes on a laptop.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(ranger)
  library(pROC);       library(ggplot2); library(patchwork)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
GRPCOL <- c(actives = PAL[["orange"]], `matched decoys` = PAL[["blue"]], `random decoys` = PAL[["purple"]])
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| Matrix",     as.character(packageVersion("Matrix")),
    "| ranger",     as.character(packageVersion("ranger")),
    "| pROC",       as.character(packageVersion("pROC")),
    "| ggplot2",    as.character(packageVersion("ggplot2")), "\n")



The versions are printed so that a reader who obtains a different number knows where to look first; the analysis is
base R<sup class="c"><a href="#r26">[26]</a></sup> with `ggplot2`<sup class="c"><a href="#r35">[35]</a></sup> figures.
One R-specific warning before anything else: `pROC` masks `stats::cov` and `stats::var`, so every covariance below is written `stats::cov(...)` explicitly. Silent masking of a base function by an attached package is one of the most common sources of irreproducible R analyses.

## 1. What a virtual screen produces, and what this unit does with it

A screening campaign has three stages. A library is assembled — purchasable catalogue, in-house collection or virtual
enumeration. A *score* is then computed for every molecule: either **ligand-based**, comparing the candidate with
molecules already known to act on the target, or **structure-based**, estimating how well it fits a pocket of the
protein. Finally — the only stage that costs money — a handful of top-ranked molecules are bought and assayed. The
computational part therefore produces one deliverable: an ordering of the library, plus the numbers that justify
trusting its top.

Everything in this unit happens after the score exists: the campaign becomes a table with a score and a ground-truth
label, and we ask how to measure that the ordering is good, how uncertain that measurement is, what makes it optimistic,
and what it predicts about a prospective experiment. The screening programs themselves (docking engines, pharmacophore
searches, shape overlays) are taught in the bioinformatics course; their output has the same form, so the analytics
transfer unchanged<sup class="c"><a href="#r34">[34]</a></sup>.

<div class="box bridge"><h4>Domain context</h4>
<p>A <b>target</b> is the protein whose activity a drug should change; this unit uses <b>ROCK1</b> (Rho-associated
protein kinase 1), an enzyme the group studies, as the target. A compound is called an <b>active</b> when a laboratory
assay shows that it inhibits the target at a low concentration; potency is reported as <b>pActivity</b> = -log<sub>10</sub>
of the molar concentration producing the effect, so pActivity 7 means 100 nM and each unit is a factor of ten. A
<b>decoy</b> is a molecule assumed inactive against the target, included in a benchmark so that a ranking has something
to be wrong about. A <b>fingerprint</b> is a binary vector in which each bit records the presence of a small
substructure, and the <b>Tanimoto coefficient</b> between two fingerprints is the number of bits they share divided by
the number of bits either has — a similarity between 0 and 1. A <b>scaffold</b> is the ring system of a molecule once
its side chains are stripped (the Bemis-Murcko framework<sup class="c"><a href="#r20">[20]</a></sup>); compounds sharing
a scaffold are usually analogues from the same medicinal-chemistry programme. <b>Enrichment</b> is the central word of
the unit: how many more actives the top of a ranking contains than a random selection of the same size.</p></div>

The frozen benchmark is `screening_library.csv`: 1,327 ROCK1 actives (curated ChEMBL
measurements<sup class="c"><a href="#r23">[23]</a></sup>, pActivity at least 6.5), 6,923 decoys matched to them in
physico-chemical property space following the DUD-E logic<sup class="c"><a href="#r2">[2]</a></sup>, and 3,768 decoys
drawn at random from the same pool of molecules measured at eight unrelated non-kinase targets, all filtered to
Tanimoto similarity below 0.35 against every known ROCK1 ligand. Two decoy sets are frozen on purpose.



In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp <- function(f) data_file(f)      # knitr runs with cwd = this unit's folder

lib <- fread(dp("screening_library.csv"))
lib[, group := ifelse(label == 1, "active", paste0("decoy: ", decoy_set))]
comp <- lib[, .(n = .N, MolWt = mean(MolWt), cLogP = mean(MolLogP), TPSA = mean(TPSA),
                HBD = mean(HBD), RotB = mean(RotB), HeavyAtoms = mean(HeavyAtoms),
                scaffolds = uniqueN(scaffold_smiles)), by = group][order(group)]
cat(sprintf("library: %d rows x %d columns | active fraction of the whole file %.3f\n",
            nrow(lib), ncol(lib), mean(lib$label)))
knitr::kable(comp, digits = 2)



The table shows the two design decisions that will drive every later result. The matched decoys look like the actives:
mean molecular weight 405 against 379 Da, mean cLogP 3.41 against 3.44, mean topological polar surface area (TPSA)
80 against 85 square angstroms. The random decoys do not: their weight spreads wider, their polarity is markedly lower
(TPSA 60) and their rotatable-bond count higher (6.2 against 5.1). A ranking that separates actives from random decoys
may be detecting the target pharmacology, or it may be detecting that ROCK1 inhibitors are kinase-like molecules with a
particular size and polarity profile — the two explanations are confounded, and only the matched set partially
decouples them. Note also the scaffold counts: 1,327 actives share only 522 Bemis-Murcko scaffolds, so the actives are
*analogue-rich*, a fact section 9 will turn into a measurable effect.

The second decision is the **active fraction**, 0.110 in this file. A real screening library has one active in a
thousand or in ten thousand<sup class="c"><a href="#r30">[30]</a></sup>. An 11 % benchmark is convenient (every metric is
stable because there are many actives) and dangerous (several metrics saturate, and the base-rate arithmetic of a
prospective campaign is completely different). Section 12 subsamples the actives to put that right.



In [ ]:
PROPS <- c("MolWt","MolLogP","TPSA","HBD","HBA","RotB","HeavyAtoms",
           "AromaticRings","RingCount","FractionCSP3","QED","formal_charge")
lib[, grp3 := factor(fifelse(label == 1, "actives",
                     fifelse(decoy_set == "property_matched", "matched decoys", "random decoys")),
                     levels = names(GRPCOL))]
long <- melt(lib[, c("grp3", "MolWt", "MolLogP", "TPSA", "RotB"), with = FALSE], id.vars = "grp3")
p1 <- ggplot(long, aes(value, colour = grp3)) +
  geom_freqpoly(aes(y = after_stat(density)), bins = 45, linewidth = 0.55) +
  facet_wrap(~ variable, nrow = 1, scales = "free") +
  scale_colour_manual(values = GRPCOL, name = NULL) +
  labs(title = "Property distributions: actives, property-matched decoys, random decoys",
       x = NULL, y = "density") +
  theme(legend.position = "bottom", axis.text.y = element_blank())
print(p1)



<p class="caption"><b>Figure 1.</b> Matching works where it was applied. The matched-decoy curves (blue) track the
actives (orange) closely in molecular weight and cLogP and reasonably in TPSA, while the random decoys (purple) are
visibly shifted — a long tail of very polar molecules, a broader weight distribution and more rotatable bonds. No
matching was applied to properties outside the matching vector, so residual differences survive even in the blue
curves; the diagnostic in section 11 measures exactly how much.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Add <code>FractionCSP3</code> and <code>QED</code> as a fifth and sixth panel of Figure 1 (one line: extend the
column list given to <code>melt()</code>). These two descriptors were <i>not</i> part of the matching vector (the
matching used molecular weight, cLogP, TPSA, hydrogen-bond donors and acceptors, rotatable bonds and formal charge).
Predict before plotting whether the matched decoys will overlap the actives on them, then check. Keep your answer:
section 11 shows that a random forest puts exactly those descriptors among the first it uses when asked to tell actives
from matched decoys.</p></div>

## 2. Fingerprints and the Tanimoto coefficient, by hand and by code

A ligand-based score needs a notion of molecular similarity. The standard one in the laboratory is the Tanimoto
coefficient over Morgan (extended-connectivity) fingerprints<sup class="c"><a href="#r10">[10]</a></sup>: each atom
environment up to a radius of two bonds is hashed into one of 2,048 bit positions, so a molecule becomes a sparse binary
vector of substructure flags. For two such vectors with \(a\) bits set in the first, \(b\) in the second and \(c\) in
common,

\[ T = \frac{c}{a + b - c}. \]

The denominator is the number of bits set in at least one of the two vectors, so \(T = 1\) only for molecules with
identical bit patterns and \(T = 0\) when they share no environment at all. Values are not calibrated across fingerprint
types: 0.35 is a weak resemblance for Morgan bits and a near-identity for some other fingerprints, which is why a
similarity threshold must always be reported together with the fingerprint that produced
it<sup class="c"><a href="#r11">[11]</a></sup>.

<div class="box bridge"><h4>Substitution — where the fingerprints come from</h4>
<p>R has no RDKit binding in this environment (<code>rcdk</code> is not installable here), so the Morgan fingerprints
(radius 2, 2,048 bits) are <strong>not recomputed in this document</strong>: they are read from the frozen sparse file
<code>data/screening_morgan_sparse.csv.gz</code>, two columns <code>id</code> (InChIKey) and <code>bit</code> (0-based
index of a set bit), and rebuilt as a sparse 0/1 matrix with <code>Matrix::sparseMatrix()</code>. The file was generated
with RDKit 2026.03.4 with exactly the generator call the Python edition makes in-notebook;
<a href="../../data/manifest_fingerprints.json">data/manifest_fingerprints.json</a> records that call and the SHA-256 of
each file, and <code>data/rock1_curated.sdf.gz</code> ships the ROCK1 structures so that a reader with ChemmineR or
rcdk can recompute the bits and check them. Everything else about similarity — the coefficient itself, the k-nearest
neighbour score, the diversity filter — is computed here in three lines of sparse-matrix algebra, which is the point:
the similarity used in every ranking below is a ratio of two integers a student can count, not an opaque library call.</p></div>



In [ ]:
sp <- fread(dp("screening_morgan_sparse.csv.gz"))         # frozen Morgan bits: (InChIKey, bit)
FP <- sparseMatrix(i = match(sp$id, lib$inchikey), j = sp$bit + 1L, x = 1,
                   dims = c(nrow(lib), 2048))
lab <- lib$label
X   <- as.matrix(lib[, ..PROPS])
tani <- function(A, B) {                                   # Tanimoto of every row of A against every row of B
  C <- as.matrix(tcrossprod(A, B))                         # shared bits c
  a <- rowSums(A); b <- rowSums(B)
  C / (outer(a, b, "+") - C)                               # c / (a + b - c)
}
cat(sprintf("fingerprints: %d molecules x %d bits | set bits per molecule: actives %.1f, decoys %.1f\n",
            nrow(FP), ncol(FP), mean(rowSums(FP)[lab == 1]), mean(rowSums(FP)[lab == 0])))



About 2.6 % of the bits of a molecule are set, which is why the matrix is stored sparse: the dense form
would be 12,018 × 2,048 doubles, and `tcrossprod` on the sparse form computes all shared-bit counts between two sets of
molecules in one call.

### 2.1 A worked example done by hand

The Python edition traces the coefficient on phenol and aniline with a 64-bit radius-1 fingerprint built on the spot;
without RDKit we instead take two real library members with few set bits and count their bits explicitly. The next
chunk picks the first pair of molecules with 18–28 set bits whose similarity falls between 0.30 and 0.55, prints both
bit lists, and lets us do the arithmetic by hand before the function does it.



In [ ]:
nb   <- rowSums(FP)
cand <- which(nb >= 18 & nb <= 28)                          # molecules small enough to write out
Tcc  <- tani(FP[cand, ], FP[cand, ]); diag(Tcc) <- 0
w    <- which(Tcc > 0.3 & Tcc < 0.55, arr.ind = TRUE)
A <- cand[w[1, 1]]; B <- cand[w[1, 2]]
bitsA <- which(FP[A, ] > 0) - 1L; bitsB <- which(FP[B, ] > 0) - 1L
shared <- intersect(bitsA, bitsB)
cat("molecule A", lib$inchikey[A], "-", lib$std_smiles[A], "\n")
cat("molecule B", lib$inchikey[B], "-", lib$std_smiles[B], "\n")
cat("a =", length(bitsA), "| b =", length(bitsB), "| c =", length(shared), "\n")
cat("shared bits:", paste(shared, collapse = ", "), "\n")
cat(sprintf("T by hand        : c / (a + b - c) = %d / %d = %.4f\n", length(shared),
            length(bitsA) + length(bitsB) - length(shared),
            length(shared) / (length(bitsA) + length(bitsB) - length(shared))))
cat(sprintf("T by the function: %.4f\n", tani(FP[A, , drop = FALSE], FP[B, , drop = FALSE])[1, 1]))



The two molecules are a cinnamoyl hydroxamic acid and a biphenyl hydroxamic-acid analogue (both decoys, measured at a
histone-deacetylase target). Molecule A has \(a = 28\) set bits, molecule B has \(b = 27\), and they share \(c = 13\),
including bits 25, 650, 694, 807 and 835 — the environments of the shared hydroxamic acid and of the *para*-substituted
benzene ring. By hand,

\[ T = \frac{13}{28 + 27 - 13} = \frac{13}{42} = 0.3095, \]

and the three lines of sparse algebra return 0.3095. The hand calculation and the code agree to the last digit printed,
which is the point of the exercise. Note how modest a similarity of 0.31 is in structural terms: two molecules sharing a
functional group and a ring already reach it, which is why 0.35 was chosen as the exclusion threshold when the decoy
pool was filtered.



In [ ]:
set.seed(0)
probe <- sample(which(lab == 1), 40)                        # 40 reference actives, for the picture only
Tp    <- tani(FP, FP[probe, ])
grp   <- list(actives = which(lab == 1),
              `matched decoys` = which(lib$decoy_set == "property_matched"),
              `random decoys`  = which(lib$decoy_set == "random"))
simdf <- rbindlist(lapply(names(grp), function(k)
  data.table(grp3 = k, maxsim = apply(Tp[grp[[k]], , drop = FALSE], 1, max))))
simdf[, .(median = median(maxsim), p95 = quantile(maxsim, 0.95), max = max(maxsim)), by = grp3]


In [ ]:
p2 <- ggplot(simdf, aes(maxsim, colour = factor(grp3, levels = names(GRPCOL)))) +
  geom_freqpoly(aes(y = after_stat(density)), bins = 50, linewidth = 0.6) +
  geom_vline(xintercept = 0.35, linetype = "dotted") +
  annotate("text", x = 0.37, y = 7, label = "0.35: decoy exclusion rule", hjust = 0, size = 2.6) +
  scale_colour_manual(values = GRPCOL, name = NULL) +
  labs(title = "The signal a similarity ranking exploits",
       x = "maximum Tanimoto similarity to 40 reference actives", y = "density") +
  theme(legend.position = c(0.75, 0.6))
print(p2)



<p class="caption"><b>Figure 2.</b> Maximum Tanimoto similarity to 40 reference actives. Actives (orange) reach a
median of 0.462 with a tail to 1.0 — analogues of the probes themselves — while the matched and random decoys sit at
0.189 and 0.173, with 95th percentiles of 0.253 and 0.233 and maxima of 0.345 and 0.300, both below the 0.35 exclusion
threshold applied when the library was frozen. This separation is what every similarity ranking converts into
enrichment; matched and random decoys are nearly indistinguishable in <i>this</i> projection, so the difference they
make to the metrics must come from elsewhere.</p>

## 3. Four ligand-based rankings

A ranking must be computable from information available *before* the assay. We therefore split the library into a
training part, which plays the role of what the laboratory already knows, and an evaluation part, which plays the role
of the untested library. Forty per cent of the actives and forty per cent of each decoy set go to training; the
remaining sixty per cent are ranked and scored. All four rankings see exactly the same training information, so the
comparison between them is fair.

<ol>
<li><b>Maximum Tanimoto similarity</b> to any training active — the oldest and still the strongest ligand-based
baseline<sup class="c"><a href="#r6">[6]</a></sup>.</li>
<li><b>Fingerprint k-nearest-neighbour score</b>: the mean Tanimoto to the five most similar training actives, which
suppresses the influence of a single lucky analogue.</li>
<li><b>QSAR probability</b>: a random forest<sup class="c"><a href="#r24">[24]</a></sup> trained here on the twelve
physico-chemical descriptors of training actives and training decoys, scoring each evaluation molecule by its predicted
probability of being active.</li>
<li><b>Descriptor distance</b>: minus the Mahalanobis distance from the molecule to the training actives in standardised
descriptor space — a one-class score that uses no decoy information at all.</li>
</ol>

The four differ in what they can see. The first two use substructure identity, the third uses whole-molecule properties
with decoy supervision, the fourth uses whole-molecule properties without it. Keeping a deliberately weak member in the
panel is useful: the consensus methods of section 13 will have to cope with it. Two R notes: `ranger` replaces
scikit-learn's `RandomForestClassifier` (it is the fast C++ random forest of the R
ecosystem<sup class="c"><a href="#r24">[24]</a></sup>, with `probability = TRUE` for class probabilities), and the
Mahalanobis distance is base R's `mahalanobis()`, so neither needs to be written by hand.



In [ ]:
set.seed(0)
idxg  <- list(active  = which(lab == 1),
              matched = which(lib$decoy_set == "property_matched"),
              random  = which(lib$decoy_set == "random"))
train <- list(); test <- list()
for (k in names(idxg)) {
  perm <- sample(idxg[[k]]); cut <- as.integer(0.4 * length(perm))
  train[[k]] <- perm[seq_len(cut)]; test[[k]] <- perm[(cut + 1):length(perm)]
}
tr_act <- train$active
mu_sc <- colMeans(X[c(tr_act, train$matched), ]); sd_sc <- apply(X[c(tr_act, train$matched), ], 2, sd)
Xs <- scale(X, center = mu_sc, scale = sd_sc)
ev_m <- c(test$active, test$matched); ev_r <- c(test$active, test$random)
y_m  <- lab[ev_m]; y_r <- lab[ev_r]
Tm_full <- tani(FP[ev_m, ], FP[tr_act, ])      # evaluation x training-active similarity matrices
Tr_full <- tani(FP[ev_r, ], FP[tr_act, ])
cat(sprintf("%s: %d train / %d test\n", names(idxg), lengths(train), lengths(test)), sep = "")
cat(sprintf("evaluation set (matched decoys): %d molecules, %d actives, prevalence %.4f\n",
            length(y_m), sum(y_m), mean(y_m)))
cat(sprintf("evaluation set (random decoys) : %d molecules, %d actives, prevalence %.4f\n",
            length(y_r), sum(y_r), mean(y_r)))


In [ ]:
build_rankings <- function(Tfull, eval_idx, decoy_key) {
  s_max <- apply(Tfull, 1, max)
  s_knn <- apply(Tfull, 1, function(v) mean(sort(v, decreasing = TRUE)[1:5]))
  tri <- c(tr_act, train[[decoy_key]])
  rf  <- ranger(y ~ ., data = data.frame(X[tri, ], y = factor(lab[tri])), num.trees = 300,
                min.node.size = 2, probability = TRUE, seed = 0, num.threads = 2)
  s_qsar <- predict(rf, data.frame(X[eval_idx, ]))$predictions[, "1"]
  Cv <- stats::cov(Xs[tr_act, ]) + diag(1e-6, length(PROPS))     # pROC masks stats::cov
  s_dist <- -sqrt(mahalanobis(Xs[eval_idx, ], colMeans(Xs[tr_act, ]), Cv))
  data.frame(max_tanimoto = s_max, knn5 = s_knn, qsar_rf = s_qsar, desc_dist = s_dist)
}
S_m <- build_rankings(Tm_full, ev_m, "matched")
S_r <- build_rankings(Tr_full, ev_r, "random")
knitr::kable(t(sapply(S_m, summary)), digits = 3,
             caption = "Score summaries on the matched-decoy evaluation set")



The evaluation set against matched decoys holds 797 actives and 4,154 decoys, a prevalence of 0.1610; against random
decoys the same 797 actives face 2,261 decoys, a prevalence of 0.2606. Those two numbers must be carried through the
rest of the unit, because three of the four metrics below depend on them. The score summaries already show the
difficulty of comparing the four columns directly: maximum Tanimoto has a median of 0.246 and a maximum of 1.0, the
random-forest probability runs from 0 to 1 with a median of 0.042, and the descriptor distance is a negative number
between -20.3 and -1.6.



In [ ]:
sdf <- melt(data.table(S_m, y = factor(y_m, labels = c("decoys", "actives"))), id.vars = "y")
p3 <- ggplot(sdf, aes(value, colour = y)) +
  geom_freqpoly(aes(y = after_stat(density)), bins = 40, linewidth = 0.55) +
  facet_wrap(~ variable, nrow = 1, scales = "free") +
  scale_colour_manual(values = c(decoys = PAL[["blue"]], actives = PAL[["orange"]]), name = NULL) +
  labs(title = "Score distributions on the evaluation set (matched decoys)", x = NULL, y = "density") +
  theme(legend.position = "bottom", axis.text.y = element_blank())
print(p3)



<p class="caption"><b>Figure 3.</b> The four scores have completely different shapes. The two similarity scores are
concentrated near 0.20-0.25 for decoys with an orange tail of analogues stretching to 1.0; the random-forest
probability is pushed towards 0 and 1 by the trees; the descriptor distance is a smooth unimodal score with heavy
overlap between the classes. Shapes matter for consensus — averaging raw scores of such different scales is
meaningless, which is why section 13 standardises or ranks them first — and they matter for thresholding, because only
the random-forest score carries anything resembling a probability.</p>

## 4. The ROC curve and its area, from first principles

Sort the evaluation set by decreasing score and walk down the list. At every position, count how many actives have
been seen (the **true-positive rate**, TPR: actives above the cut divided by all actives) and how many decoys have been
seen (the **false-positive rate**, FPR). Plotting TPR against FPR as the cut slides from the top of the list to the
bottom traces the **receiver-operating-characteristic (ROC) curve**<sup class="c"><a href="#r27">[27]</a></sup>. A
useless ranking produces the diagonal; a perfect one rises to the top-left corner.

The area under that curve (**ROC-AUC**) has a probabilistic meaning worth memorising: it is the probability that a
randomly chosen active is ranked above a randomly chosen decoy. That interpretation also gives the cheapest way to
compute it, the Mann-Whitney statistic

\[ \mathrm{AUC} = \frac{\sum_{i \in \text{actives}} r_i - n_1 (n_1 + 1)/2}{n_1 n_0}, \]

where \(r_i\) is the rank of active \(i\) when scores are sorted in increasing order, \(n_1\) is the number of actives
and \(n_0\) the number of decoys. The next chunk implements the curve and the formula with base R's `rank()` and
`order()`, and checks the result against `pROC`<sup class="c"><a href="#r25">[25]</a></sup>.



In [ ]:
roc_auc <- function(y, s) {                       # Mann-Whitney form; rank() averages ties
  r <- rank(s); n1 <- sum(y); n0 <- length(y) - n1
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}
roc_pts <- function(y, s) {                       # TPR and FPR at every cut of the score-sorted list
  ys <- y[order(-s, method = "radix")]
  list(fpr = c(0, cumsum(1 - ys) / sum(1 - ys)), tpr = c(0, cumsum(ys) / sum(ys)))
}
for (c_ in c("max_tanimoto", "desc_dist"))
  cat(sprintf("%-13s ours %.6f | pROC %.6f\n", c_, roc_auc(y_m, S_m[[c_]]),
              as.numeric(pROC::auc(pROC::roc(y_m, S_m[[c_]], quiet = TRUE, direction = "<")))))
for (c_ in names(S_m)) cat(sprintf("%-13s AUC = %.3f\n", c_, roc_auc(y_m, S_m[[c_]])))



Our eight-line implementation and `pROC` agree to six decimals on both the best and the worst ranking, so the rest of
the unit can use the fast version with confidence. The areas place maximum Tanimoto similarity (0.993) and the
five-nearest-neighbour score (0.991) above the property-based random forest (0.968) and far above the descriptor
distance (0.867).



In [ ]:
roc_long <- rbindlist(lapply(names(S_m), function(c_) {
  p <- roc_pts(y_m, S_m[[c_]])
  data.table(ranking = sprintf("%s (AUC %.3f)", c_, roc_auc(y_m, S_m[[c_]])),
             fpr = p$fpr, tpr = p$tpr)
}))
RC <- setNames(unname(PAL), names(S_m))
cols <- setNames(unname(PAL), unique(roc_long$ranking))
pa <- ggplot(roc_long, aes(fpr, tpr, colour = ranking)) + geom_line(linewidth = 0.6) +
  geom_abline(linetype = "dotted") + scale_colour_manual(values = cols, name = NULL) +
  labs(title = "ROC, linear axes", x = "false-positive rate", y = "true-positive rate") +
  theme(legend.position = c(0.62, 0.25), legend.text = element_text(size = 6))
pb <- ggplot(roc_long[fpr > 0], aes(fpr, tpr, colour = ranking)) + geom_line(linewidth = 0.6) +
  scale_x_log10() + scale_colour_manual(values = cols, guide = "none") +
  labs(title = "ROC, semi-logarithmic: the part we can afford to test",
       x = "false-positive rate (log)", y = "true-positive rate")
print(pa | pb)



<p class="caption"><b>Figure 4.</b> ROC curves of the four rankings from the cumulative-count definition (left) and on
a logarithmic false-positive axis (right). The left panel is dominated by the bulk of the list, where no compound will
ever be bought; the right panel magnifies the first per mille of decoys, where the decision is made. The two similarity
rankings (AUC 0.993 and 0.991) and the QSAR probability (0.968) separate more clearly there than their areas suggest,
and the descriptor distance (0.867) is the weakest everywhere.</p>

The semi-logarithmic panel is the first warning about ROC-AUC in this setting: the area integrates over the whole
ranking, giving equal weight to an active recovered at position 5 and one recovered at position 5,000. For a campaign
that will test fifty compounds, only the first fraction of a per cent of the list matters. The metrics of the next
three sections exist to repair that mismatch<sup class="c"><a href="#r1">[1]</a></sup>.

## 5. Enrichment factors — and the ceiling nobody mentions

The **enrichment factor** at a fraction \(f\) of the library is the most quoted number in the screening literature:

\[ \mathrm{EF}_f = \frac{\text{actives found in the top } f}{\text{compounds in the top } f} \Big/ \frac{n_1}{N}, \]

the hit rate in the top \(f\) divided by the hit rate of the whole library. EF = 1 means the ranking is worthless;
EF = 10 means the top of the list is ten times richer in actives than the library as a whole.

What is rarely said is that EF has a hard ceiling. If the top \(fN\) positions are *all* actives, the hit rate there is
1 and \(\mathrm{EF}_f = N / n_1 = 1/\text{prevalence}\). With this evaluation set's prevalence of 0.1610, no ranking can
score above 6.212 at any fraction — and if \(f\) is smaller than the prevalence, a perfect ranking fills the entire top
fraction with actives and attains that ceiling exactly.



In [ ]:
ef <- function(y, s, frac) {
  k <- max(1, round(frac * length(y)))
  mean(y[order(-s, method = "radix")][1:k]) / mean(y)
}
ef_tab <- data.frame(AUC  = sapply(S_m, function(s) roc_auc(y_m, s)),
                     EF1  = sapply(S_m, function(s) ef(y_m, s, 0.01)),
                     EF5  = sapply(S_m, function(s) ef(y_m, s, 0.05)),
                     EF20 = sapply(S_m, function(s) ef(y_m, s, 0.20)))
cat(sprintf("EF ceiling at this prevalence = 1/%.4f = %.3f\n", mean(y_m), 1 / mean(y_m)))
knitr::kable(ef_tab, digits = 3)



Three of the four rankings hit the ceiling 6.212 at 1 % *and* at 5 % — their top 50 and their top 248 compounds are pure
actives — so at those fractions the metric cannot distinguish an excellent ranking from a merely very good one.
Reported without the ceiling, "EF<sub>1%</sub> = 6.2" sounds modest; in fact it is a perfect score for a benchmark of
this composition. Reported without the prevalence, the same number is uninterpretable. **Always publish the active
fraction of the evaluation set next to any enrichment factor**, and prefer a fraction \(f\) well above the prevalence
(here 20 %, where the rankings do separate: 4.89 for maximum Tanimoto, 4.84 for the k-nearest-neighbour score, 4.44 for
the random forest and 3.38 for the descriptor distance) or a metric that does not saturate.



In [ ]:
acc <- rbindlist(lapply(names(S_m), function(c_) {
  ys <- y_m[order(-S_m[[c_]], method = "radix")]
  data.table(ranking = c_, screened = seq_along(ys) / length(ys), found = cumsum(ys) / sum(ys))
}))
gridf <- seq(0.001, 1, length.out = 400)
p5 <- ggplot(acc, aes(screened, found, colour = ranking)) + geom_line(linewidth = 0.6) +
  geom_line(data = data.table(screened = gridf, found = gridf), colour = "grey35",
            linetype = "dotted", inherit.aes = FALSE, aes(screened, found)) +
  geom_line(data = data.table(screened = gridf, found = pmin(gridf / mean(y_m), 1)), colour = "grey35",
            linetype = "dashed", inherit.aes = FALSE, aes(screened, found)) +
  scale_colour_manual(values = RC, name = NULL) +
  labs(title = "Accumulation curves, with the random and perfect reference lines",
       x = "fraction of the library screened", y = "fraction of actives recovered") +
  theme(legend.position = c(0.72, 0.3))
print(p5)



<p class="caption"><b>Figure 5.</b> Accumulation curve — actives recovered against library screened — with the random
diagonal (dotted) and the perfect-ranking envelope (dashed, initial slope 1/prevalence = 6.21). The similarity rankings
follow the envelope for the first few per cent and have recovered all 797 actives by about a third of the list; the
descriptor distance hugs the diagonal in the early region that matters. An enrichment factor is the height of such a
curve at one abscissa, divided by that abscissa.</p>

## 6. BEDROC and RIE: putting the weight where the money is

Truchon and Bayly proposed a metric that formalises the "early recognition"
preference<sup class="c"><a href="#r1">[1]</a></sup>. Give the active at relative rank \(x = r/N\) an exponentially
decaying weight \(e^{-\alpha x}\), so that an active at the very top counts fully and one halfway down the list counts
for almost nothing. The **robust initial enhancement** is the sum of those weights divided by its expectation under a
random ranking,

\[ \mathrm{RIE} = \frac{\frac{1}{n_1}\sum_{i \in \text{actives}} e^{-\alpha r_i / N}}
{\frac{1}{N}\,\frac{1 - e^{-\alpha}}{e^{\alpha/N} - 1}}, \]

and **BEDROC** is RIE rescaled to the interval \([0, 1]\) so that 0 is the worst and 1 the best achievable ranking at
the given prevalence \(R_a = n_1/N\):

\[ \mathrm{BEDROC} = \mathrm{RIE}\cdot\frac{R_a \sinh(\alpha/2)}{\cosh(\alpha/2) - \cosh(\alpha/2 - \alpha R_a)} + \frac{1}{1 - e^{\alpha(1 - R_a)}}. \]

The constant \(\alpha\) sets how early "early" is: the weight falls to \(1/e\) at \(x = 1/\alpha\) and about 80 % of it
lies in the top \(4/\alpha\) of the list, so the conventional \(\alpha = 20\) concentrates on the top 20 % and
\(\alpha = 160\) on the top few per mille. Choosing \(\alpha\) after seeing the results is a form of multiple testing;
fix it from the size of the campaign (test 50 of 10,000 compounds, \(f = 0.005\), \(\alpha \approx 4/f = 800\)),
accepting that so few actives in the top make the estimate noisy.



In [ ]:
rie_bedroc <- function(y, s, alpha = 20) {        # Truchon & Bayly (2007), from the definitions
  ys <- y[order(-s, method = "radix")]
  N <- length(ys); n1 <- sum(ys); Ra <- n1 / N; r <- which(ys == 1)
  rie <- (sum(exp(-alpha * r / N)) / n1) / ((1 / N) * (1 - exp(-alpha)) / (exp(alpha / N) - 1))
  sc  <- Ra * sinh(alpha / 2) / (cosh(alpha / 2) - cosh(alpha / 2 - alpha * Ra))
  c(RIE = rie, BEDROC = unname(rie * sc + 1 / (1 - exp(alpha * (1 - Ra)))))
}
set.seed(1)
cat(sprintf("sanity: perfect ranking BEDROC %.3f | random ranking BEDROC %.3f (prevalence %.3f)\n",
            rie_bedroc(y_m, y_m + seq(0, 1e-6, length.out = length(y_m)))[2],
            rie_bedroc(y_m, runif(length(y_m)))[2], mean(y_m)))
bed <- t(sapply(S_m, function(s) c(`RIE(a=20)` = unname(rie_bedroc(y_m, s, 20)[1]),
                                   `BEDROC(a=20)` = rie_bedroc(y_m, s, 20)[2],
                                   `BEDROC(a=8)`  = rie_bedroc(y_m, s, 8)[2],
                                   `BEDROC(a=160)` = rie_bedroc(y_m, s, 160)[2])))
knitr::kable(bed, digits = 3)



The sanity check is the first thing to read: a ranking that puts every active first scores BEDROC 1.000 and a uniform
random score scores 0.160, which is the prevalence — BEDROC is calibrated so that a useless ranking returns \(R_a\),
not zero. The four rankings then score 0.997, 0.995, 0.950 and 0.776 at \(\alpha = 20\); the corresponding RIE values
are 5.94, 5.93, 5.67 and 4.63, i.e. the early-weighted density of actives is about six times the random expectation for
the similarity rankings.



In [ ]:
wdf <- rbindlist(lapply(c(8, 20, 80, 160), function(a)
  data.table(alpha = factor(a), x = seq(0, 0.3, length.out = 300), w = exp(-a * seq(0, 0.3, length.out = 300)))))
pw <- ggplot(wdf, aes(x, w, colour = alpha)) + geom_line(linewidth = 0.6) +
  scale_colour_manual(values = unname(PAL), name = "alpha") +
  labs(title = "The BEDROC weight function", x = "relative rank r/N", y = "weight exp(-alpha r/N)")
alphas <- c(2, 5, 8, 20, 40, 80, 160, 321)
sweep <- rbindlist(lapply(names(S_m), function(c_)
  data.table(ranking = c_, alpha = alphas,
             bedroc = sapply(alphas, function(a) rie_bedroc(y_m, S_m[[c_]], a)[2]))))
ps <- ggplot(sweep, aes(alpha, bedroc, colour = ranking)) + geom_line(linewidth = 0.6) +
  geom_point(size = 1) + scale_x_log10() + scale_colour_manual(values = RC, name = NULL) +
  labs(title = "Ranking order can depend on the choice of alpha", x = "alpha", y = "BEDROC") +
  theme(legend.position = c(0.75, 0.35), legend.text = element_text(size = 6))
print(pw | ps)



<p class="caption"><b>Figure 6.</b> Left: the exponential weight for four values of alpha; at alpha = 20 a compound at
10 % of the list still carries 14 % of the weight of the first compound, at alpha = 160 it carries none. Right: BEDROC
of the four rankings as alpha grows. At alpha = 8 they read 0.991, 0.987, 0.929 and 0.742, and by alpha = 160 the first
three have reached 1.000 while the descriptor distance reaches 0.929 — at a 16 % active fraction the extreme top of
every ranking is pure actives, so a very early-weighted metric stops discriminating exactly as the enrichment factor
did. A reviewer who sees only "BEDROC = 0.99" cannot tell which regime produced it: report alpha, and report it as a
decision taken before the analysis.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Compute BEDROC at alpha = 20 for a ranking that places all actives between positions 10 % and 20 % of the list and
all decoys elsewhere, and for one that places them between 0 % and 10 %. Both have the same number of actives in the
top 20 %, and almost the same ROC-AUC. How different are their BEDROC values, and which of the two would you rather
have bought? Build the two artificial score vectors with <code>order()</code> and
<code>rie_bedroc()</code>; three lines of R are enough.</p></div>

## 7. Hit rate at k: the number a medicinal chemist asks for

The metrics so far are dimensionless. The question actually asked in a project meeting is different: *if I buy the top
k compounds, how many will work?* That is the **hit rate at k**, the fraction of actives among the first k entries, and
its practical companion, the expected number of hits \(k \times \text{hit rate}\). It depends on the prevalence of the
evaluation set, which is precisely why it is the honest metric to carry into section 12.



In [ ]:
ks <- unique(round(10 ^ seq(0, log10(length(y_m)), length.out = 60)))
hr <- rbindlist(lapply(names(S_m), function(c_) {
  ys <- y_m[order(-S_m[[c_]], method = "radix")]
  data.table(ranking = c_, k = ks, hit_rate = sapply(ks, function(k) mean(ys[1:k])))
}))
p7 <- ggplot(hr, aes(k, hit_rate, colour = ranking)) + geom_line(linewidth = 0.6) + scale_x_log10() +
  geom_hline(yintercept = mean(y_m), linetype = "dotted") +
  annotate("text", x = 2, y = mean(y_m) + 0.04, label = "library hit rate (prevalence)", hjust = 0, size = 2.6) +
  scale_colour_manual(values = RC, name = NULL) +
  labs(title = "Hit rate at k on the retrospective benchmark",
       x = "k = number of compounds bought", y = "hit rate among the top k") +
  theme(legend.position = c(0.25, 0.3))
print(p7)
for (c_ in names(S_m)) {
  ys <- y_m[order(-S_m[[c_]], method = "radix")]
  cat(sprintf("%-13s hit rate at k = 50: %.2f | at k = 500: %.2f\n", c_, mean(ys[1:50]), mean(ys[1:500])))
}



<p class="caption"><b>Figure 7.</b> Hit rate against purchase size k (logarithmic). Three rankings return 100 % hits
for the first fifty compounds and the descriptor distance 96 %; at k = 500 the two similarity rankings still return
100 %, the random forest 94 % and the descriptor distance 75 %. All curves decay to the dotted prevalence line as k
approaches the library size. These are the spectacular retrospective numbers that section 12 deflates.</p>

## 8. Confidence intervals: how much of this is luck?

Every number printed so far is an estimate from one finite sample of actives and one finite sample of decoys. Replace
the sample and the number moves. The standard way to quantify that movement without any distributional assumption is
the **bootstrap**<sup class="c"><a href="#r28">[28]</a></sup>: resample the data with replacement many times, recompute
the statistic each time, and read the 2.5th and 97.5th percentiles of the resulting distribution as a 95 % interval.

For a screening benchmark the resampling must be **stratified**: draw \(n_1\) actives with replacement from the actives
and \(n_0\) decoys with replacement from the decoys, so that every replicate has the same composition as the original
and the prevalence — on which EF and BEDROC depend — is held fixed. Resampling the pooled list instead would make the
prevalence fluctuate and mix two sources of variation. Note what this interval does and does not cover: it covers the
sampling of molecules, not the choice of decoy set, not the choice of fingerprint, and not the choice of reference
actives. Sections 9 to 11 measure those separately, and they are larger.



In [ ]:
boot_ci <- function(y, s, stat, n_boot = 400, seed = 0) {
  set.seed(seed)                                   # an explicit seed before every resampling
  ai <- which(y == 1); di <- which(y == 0); v <- numeric(n_boot)
  for (b in seq_len(n_boot)) {
    idx <- c(sample(ai, length(ai), TRUE), sample(di, length(di), TRUE))
    v[b] <- stat(y[idx], s[idx])
  }
  c(mean = mean(v), lo = unname(quantile(v, 0.025)), hi = unname(quantile(v, 0.975)))
}
ci <- as.data.frame(t(sapply(names(S_m), function(c_) {
  s <- S_m[[c_]]
  a <- boot_ci(y_m, s, roc_auc)
  b <- boot_ci(y_m, s, function(yy, ss) rie_bedroc(yy, ss, 20)[2])
  e <- boot_ci(y_m, s, function(yy, ss) ef(yy, ss, 0.20))
  c(auc = a, bedroc20 = b, ef20 = e)
})))
knitr::kable(data.frame(
  AUC      = sprintf("%.3f [%.3f, %.3f]", ci$auc.mean, ci$auc.lo, ci$auc.hi),
  BEDROC20 = sprintf("%.3f [%.3f, %.3f]", ci$bedroc20.mean, ci$bedroc20.lo, ci$bedroc20.hi),
  EF20     = sprintf("%.2f [%.2f, %.2f]", ci$ef20.mean, ci$ef20.lo, ci$ef20.hi),
  row.names = names(S_m)))


In [ ]:
cidf <- data.table(ranking = factor(names(S_m), levels = rev(names(S_m))),
                   m = ci$auc.mean, lo = ci$auc.lo, hi = ci$auc.hi)
p8 <- ggplot(cidf, aes(m, ranking)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.18, colour = PAL[["blue"]]) +
  geom_point(size = 2, colour = PAL[["blue"]]) +
  labs(title = "Four rankings with their uncertainty", x = "ROC-AUC (95 % bootstrap interval)", y = NULL)
print(p8)



<p class="caption"><b>Figure 8.</b> The intervals are narrow — 0.993 [0.989, 0.997] for maximum Tanimoto and
0.968 [0.962, 0.974] for the random forest — because the evaluation set contains 797 actives. Those of the two
similarity rankings nearly overlap, so the apparent advantage of maximum Tanimoto over the k-nearest-neighbour score
(0.993 against 0.991) is barely supported, while the gap to the random forest and to the descriptor distance is far
outside anything sampling noise could produce. With the fifty to two hundred actives of a typical in-house benchmark
the same intervals would be several times wider, which is the usual reason published comparisons of two scoring
functions cannot be reproduced<sup class="c"><a href="#r13">[13]</a></sup>.</p>

## 9. How many actives do you know? The reference set decides the score

The maximum-Tanimoto ranking used 530 training actives. A real project at its start knows five, or twenty, or — after a
literature review of a well-studied target — a few hundred. Because the score of a candidate is the similarity to its
closest known active, the number and the identity of those references are not a detail of the protocol: they are the
protocol. The next chunk repeats the same ranking with reference sets of 5 to 530 actives, ten random draws each. No
new similarity has to be computed: the maximum over a subset of references is the row maximum over the corresponding
*columns* of the matrix `Tm_full`, which is why the whole experiment costs a fraction of a second.



In [ ]:
set.seed(3)
sizes <- c(5, 10, 25, 50, 100, 200, 530)
ref_tab <- as.data.frame(t(sapply(sizes, function(n) {
  reps <- if (n < length(tr_act)) 10 else 1
  a <- sapply(seq_len(reps), function(i) {
    ri <- sample(seq_along(tr_act), n)
    roc_auc(y_m, apply(Tm_full[, ri, drop = FALSE], 1, max))
  })
  c(n_references = n, mean_AUC = mean(a), sd = sd(a), min = min(a), max = max(a))
})))
knitr::kable(ref_tab, digits = 3)


In [ ]:
p9 <- ggplot(ref_tab, aes(n_references, mean_AUC)) +
  geom_ribbon(aes(ymin = min, ymax = max), fill = PAL[["blue"]], alpha = 0.18) +
  geom_line(colour = PAL[["blue"]], linewidth = 0.6) + geom_point(colour = PAL[["blue"]], size = 1.6) +
  scale_x_log10() +
  labs(title = "The 'method' is mostly the size of your reference set",
       x = "number of known actives used as references",
       y = "ROC-AUC of the max-Tanimoto ranking")
print(p9)



<p class="caption"><b>Figure 9.</b> ROC-AUC of one and the same algorithm as a function of how many actives it is
given: 0.835 with five references, 0.928 with twenty-five, 0.969 with a hundred and 0.993 with all 530. The shaded band
is the range over ten random draws of the reference set — with five references two groups running the identical
protocol on the identical library can report 0.75 and 0.90, and with twenty-five, 0.89 and 0.95. Any comparison of two
screening methods that does not hold the reference set fixed is comparing reference sets.</p>

This also settles a comparison worth making. The frozen library's own README reports ROC-AUC 0.917 against matched
decoys for a twenty-five-reference Tanimoto ranking (built in Python with RDKit); our ten R draws of twenty-five
references average 0.928, with individual draws from 0.891 to 0.951, so the README value sits comfortably inside our
band. The agreement is reassuring and the width of the band is the real message: a single draw of a reference set can
land several hundredths of an AUC unit away from the protocol's expectation. The enrichment factor at 1 %, by contrast,
is instead pinned by arithmetic: 6.212 against matched decoys and 3.837 against random ones, exactly the ceiling
1/prevalence in each set, with nothing left to vary. The README quotes 6.3 and 3.9 for the same quantity because its
25-reference draw leaves a slightly different prevalence in the evaluation set; both numbers are the ceiling of their
own set, which is why neither depends on the reference molecules.

## 10. Matched versus random decoys: the same method, two answers

Now change only the decoy set. The evaluation actives are identical, the training actives are identical, the scores are
computed in the same way; only the 4,154 matched decoys are replaced by 2,261 randomly drawn ones.



In [ ]:
decoy_cmp <- data.frame(
  AUC_matched    = sapply(names(S_m), function(c_) roc_auc(y_m, S_m[[c_]])),
  AUC_random     = sapply(names(S_r), function(c_) roc_auc(y_r, S_r[[c_]])),
  BEDROC_matched = sapply(names(S_m), function(c_) rie_bedroc(y_m, S_m[[c_]], 20)[2]),
  BEDROC_random  = sapply(names(S_r), function(c_) rie_bedroc(y_r, S_r[[c_]], 20)[2]),
  EF1_matched    = sapply(names(S_m), function(c_) ef(y_m, S_m[[c_]], 0.01)),
  EF1_random     = sapply(names(S_r), function(c_) ef(y_r, S_r[[c_]], 0.01)))
decoy_cmp$AUC_gain <- decoy_cmp$AUC_random - decoy_cmp$AUC_matched
knitr::kable(decoy_cmp, digits = 3)


In [ ]:
mk <- function(S, y, title) {
  d <- rbindlist(lapply(names(S), function(c_) {
    p <- roc_pts(y, S[[c_]])
    data.table(ranking = sprintf("%s (%.3f)", c_, roc_auc(y, S[[c_]])), fpr = p$fpr, tpr = p$tpr)
  }))
  ggplot(d[fpr > 0], aes(fpr, tpr, colour = ranking)) + geom_line(linewidth = 0.6) + scale_x_log10() +
    scale_colour_manual(values = setNames(unname(PAL), unique(d$ranking)), name = NULL) +
    labs(title = title, x = "false-positive rate (log)", y = "true-positive rate") +
    theme(legend.position = c(0.68, 0.28), legend.text = element_text(size = 6))
}
print(mk(S_m, y_m, "ROC against property-matched decoys") | mk(S_r, y_r, "ROC against random decoys"))



<p class="caption"><b>Figure 10.</b> The same four rankings evaluated against property-matched decoys (left) and
random decoys (right). The similarity rankings are almost unaffected (0.993 to 0.995 and 0.991 to 0.995), the QSAR
random forest gains 0.028 of AUC (0.968 to 0.997) and the descriptor distance gains 0.095 (0.867 to 0.961). The
ordering of the methods changes: against random decoys the property-based random forest becomes the best in the panel,
against matched decoys it is third.</p>

That asymmetry is the argument for property matching<sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r15">[15]</a></sup>:
a property-based score is rewarded by any decoy set whose properties differ from the actives, because the task it
actually solves is "does this look like a kinase inhibitor?" rather than "does it bind ROCK1?". Substructure similarity
is comparatively immune, the exclusion rule having already removed decoys close to known ligands. The laboratory rule:
**report every metric against the hardest decoy set you can build, and never compare methods evaluated against
different decoys**<sup class="c"><a href="#r12">[12]</a></sup><sup class="c"><a href="#r14">[14]</a></sup>. Matching is
itself imperfect — decoys matched too tightly become near-actives and punish a method for finding
them<sup class="c"><a href="#r16">[16]</a></sup> — so benchmarks built from experimentally confirmed inactives
(MUV<sup class="c"><a href="#r19">[19]</a></sup>, LIT-PCBA<sup class="c"><a href="#r18">[18]</a></sup>) are now
preferred, at the price of smaller and noisier active sets.

## 11. The artificial-enrichment diagnostic

There is a direct test of how much of a benchmark's separability lives in the properties rather than in the
pharmacology: train a classifier that is *only allowed* to see the physico-chemical descriptors, with no fingerprint
and no target information, and ask how well it distinguishes actives from decoys under honest cross-validation. If that
classifier succeeds, any method using the same descriptors can succeed for the same uninteresting
reason<sup class="c"><a href="#r3">[3]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>.



In [ ]:
cv_auc <- function(sel, seed = 0) {               # stratified 5-fold, written out
  set.seed(seed); y <- lab[sel]; fold <- integer(length(sel))
  for (cl in c(0, 1)) { i <- which(y == cl); fold[i] <- sample(rep(1:5, length.out = length(i))) }
  sapply(1:5, function(f) {
    tr <- fold != f
    m <- ranger(y ~ ., data = data.frame(X[sel[tr], ], y = factor(y[tr])), num.trees = 300,
                probability = TRUE, seed = 0, num.threads = 2)
    roc_auc(y[!tr], predict(m, data.frame(X[sel[!tr], ]))$predictions[, "1"])
  })
}
for (ds in c("property_matched", "random")) {
  a <- cv_auc(which(lib$decoy_set == ds | lab == 1))
  cat(sprintf("%-17s random forest on PROPERTIES ALONE, 5-fold ROC-AUC = %.3f +/- %.3f\n", ds, mean(a), sd(a)))
}
sel  <- which(lib$decoy_set == "property_matched" | lab == 1)
rfd  <- ranger(y ~ ., data = data.frame(X[sel, ], y = factor(lab[sel])), num.trees = 300,
               importance = "permutation", seed = 0, num.threads = 2)
imp  <- sort(ranger::importance(rfd), decreasing = TRUE)
print(round(imp, 4))


In [ ]:
impdf <- data.table(prop = factor(names(imp), levels = rev(names(imp))), imp = as.numeric(imp))
pi1 <- ggplot(impdf, aes(imp, prop)) + geom_col(fill = PAL[["blue"]], width = 0.7) +
  labs(title = "What separates actives from MATCHED decoys", x = "permutation importance", y = NULL)
pi2 <- ggplot(lib, aes(FractionCSP3, colour = grp3)) +
  geom_freqpoly(aes(y = after_stat(density)), bins = 40, linewidth = 0.6) +
  scale_colour_manual(values = GRPCOL, name = NULL) +
  labs(title = "A descriptor the matching forgot", x = "FractionCSP3 (fraction of sp3 carbons)", y = "density") +
  theme(legend.position = c(0.68, 0.75), legend.text = element_text(size = 6))
print(pi1 | pi2)



<p class="caption"><b>Figure 11.</b> Left: permutation importances of a random forest trained to tell actives from
property-matched decoys using properties only. AromaticRings (0.086), QED (0.074), MolWt (0.068), FractionCSP3 (0.067)
and TPSA (0.066) lead; of these, only molecular weight and TPSA were part of the matching vector, and
<code>formal_charge</code> — matched exactly — contributes nothing (0.000). Right: the FractionCSP3 distributions that
produce part of that importance; ROCK1 actives are flatter, more aromatic molecules than either decoy population.</p>

The diagnostic returns a 5-fold ROC-AUC of **0.978 ± 0.007** against the matched decoys and **0.999 ± 0.000** against
the random ones. Read that carefully: a model that knows nothing but twelve bulk properties reproduces, and exceeds,
the performance of every ligand-based ranking of section 3 except the two fingerprint similarities. The benchmark is
therefore not measuring pharmacology; at best it measures the ability to recognise the chemical class from which ROCK1
inhibitors were drawn. This does not make the benchmark useless — relative comparisons between methods on the *same*
decoys remain informative — but it forbids the sentence "our model achieves AUC 0.98 and has therefore learnt the
binding determinants of ROCK1".

Three defences, all used by the group: report the diagnostic alongside the results as a declared property of the
benchmark; prefer splits that break the property shortcut (a scaffold split, Unit 12) or decoys matched on more
properties; and validate prospectively, the only evaluation no benchmark artefact survives.

<div class="exercise"><h4>Exercise 3</h4>
<p>Remove <code>FractionCSP3</code>, <code>QED</code> and <code>AromaticRings</code> from <code>PROPS</code> and rerun
the diagnostic. How far does the property-only AUC against the matched decoys fall? Then add a few hundred Morgan
fingerprint bits as extra columns (<code>as.matrix(FP[, 1:300])</code>) and rerun once more. Which of the three feature
sets would you use to argue that a model has learnt something target-specific, and what would you still not be able to
claim?</p></div>

## 12. Prevalence, base rates and what the top of the list is really worth

The benchmark's active fraction is 16.1 % in the evaluation set. A purchasable screening library contains perhaps one
active per thousand molecules for a tractable target, and far fewer for a hard one<sup class="c"><a href="#r30">[30]</a></sup>.
Prevalence does not affect ROC-AUC — the probability that an active outranks a decoy is unchanged if we delete nine out
of ten actives at random — but it changes everything a project actually cares about.

### 12.1 The calculation by hand

Let the library contain \(N\) compounds of which a fraction \(\pi\) are active, and suppose our ranking puts a fraction
\(e\) of the actives into the top \(f\) of the list. The hit rate among the compounds we buy is

\[ \text{hit rate} = \frac{\pi e N}{f N} = \pi \cdot \frac{e}{f} = \pi \cdot \mathrm{EF}_f . \]

With \(\pi = 0.001\) and an enrichment factor of 96 in the top 1 %, the hit rate is \(0.001 \times 96 = 0.096\): about
one compound in ten of the top 1 % is active. Buying fifty compounds from that top 1 % gives an expected
\(50 \times 0.096 = 4.8\) hits, against \(50 \times 0.001 = 0.05\) for a random draw — a spectacular improvement that
still means forty-five failures. Written as a posterior probability (the Bayes rule of Unit 10), the probability that a
specific top-ranked compound is active is

\[ P(\text{active}\mid\text{top } f) = \frac{P(\text{top } f\mid\text{active})\,\pi}{P(\text{top } f)}
= \frac{e\,\pi}{f} = 0.096 , \]

with \(e = 0.96\) and \(f = 0.01\). The next chunk performs the experiment that this arithmetic predicts: it keeps the
decoys fixed and subsamples the actives down to prevalences of 5 %, 1 % and 0.1 %, repeating each subsample 200 times.



In [ ]:
prevalence_experiment <- function(s, y, prevs = c(0.161, 0.05, 0.01, 0.001), n_rep = 200, seed = 0) {
  set.seed(seed); ai <- which(y == 1); di <- which(y == 0)
  rbindlist(lapply(prevs, function(p) {
    n_a <- min(length(ai), max(1, round(p * length(di) / (1 - p))))
    r <- replicate(n_rep, {
      idx <- c(sample(ai, n_a), di); yy <- y[idx]; ss <- s[idx]
      c(ef(yy, ss, 0.01), ef(yy, ss, 0.05), roc_auc(yy, ss),
        mean(yy[order(-ss, method = "radix")][1:100]))
    })
    data.table(prevalence = n_a / (n_a + length(di)), n_actives = n_a, AUC = mean(r[3, ]),
               EF1 = mean(r[1, ]), EF1_ceiling = min(1 / (n_a / (n_a + length(di))), 100),
               EF5 = mean(r[2, ]), hit_rate_top100 = mean(r[4, ]),
               expected_hits_in_50 = 50 * mean(r[4, ]))
  }))
}
prev_tab <- prevalence_experiment(S_m$max_tanimoto, y_m)
knitr::kable(prev_tab, digits = 4)


In [ ]:
pp1 <- ggplot(prev_tab, aes(prevalence, AUC)) + geom_line(colour = PAL[["blue"]]) +
  geom_point(colour = PAL[["blue"]], size = 1.6) + scale_x_log10() + ylim(0.5, 1.05) +
  labs(title = "ROC-AUC is blind to prevalence", x = "active fraction", y = "ROC-AUC")
pp2 <- ggplot(prev_tab, aes(prevalence, EF1)) + geom_line(colour = PAL[["orange"]]) +
  geom_point(colour = PAL[["orange"]], size = 1.6) +
  geom_line(aes(y = EF1_ceiling), linetype = "dashed", colour = "grey35") +
  scale_x_log10() + scale_y_log10() +
  labs(title = "EF grows - but only up to its ceiling", x = "active fraction", y = "EF at 1 % (dashed: 1/prevalence)")
pp3 <- ggplot(prev_tab, aes(prevalence, hit_rate_top100)) + geom_line(colour = PAL[["green"]]) +
  geom_point(colour = PAL[["green"]], size = 1.6) +
  geom_line(aes(y = prevalence), linetype = "dotted", colour = "grey35") +
  scale_x_log10() + scale_y_log10() +
  labs(title = "Hit rate in the top 100 collapses", x = "active fraction", y = "hit rate among top 100")
print(pp1 | pp2 | pp3)



<p class="caption"><b>Figure 12.</b> One ranking, four libraries differing only in their active fraction. ROC-AUC stays
between 0.993 and 0.992 (left). The enrichment factor at 1 % rises from 6.2 to 96 (centre) only because its ceiling
rises: it stays pinned to 1/prevalence (dashed), and at 0.1 % prevalence — four actives in 4,196 compounds — it even
falls slightly below the 1 % ceiling of 100. The hit rate among the top hundred (right) falls from 100 % to 40.8 % at
1 % prevalence and to 3.9 % at 0.1 %, that is about two hits per fifty purchases, against the dotted random-draw line —
still a factor of forty better.</p>

This translation is the most useful one in the unit. "AUC 0.99, EF<sub>1%</sub> = 96" is a method at its best; the same
method on a realistic library predicts about two hits in fifty purchases — excellent for medicinal chemistry,
disappointing to anyone who read only the enrichment factor. Report both, with the prevalence you assumed.

## 13. Consensus and rank aggregation: does combining help?

When several scores are available the reflex is to combine them. Four schemes are
standard<sup class="c"><a href="#r5">[5]</a></sup>:

<ol>
<li><b>z-score consensus</b>: standardise each score to mean 0 and unit variance, then average. Keeps magnitude
information; sensitive to the shape of each score distribution.</li>
<li><b>rank-by-rank</b>: replace each score by its rank and average the ranks. Scale-free, insensitive to outliers,
discards magnitude.</li>
<li><b>rank-by-vote</b>: count in how many rankings a compound appears in the top 5 %. Very robust, extremely coarse —
most compounds get 0 votes, and ties are resolved arbitrarily.</li>
<li><b>exponential consensus ranking (ECR)</b>: sum \(e^{-r/\sigma}\) over the rankings, with \(\sigma\) the number of
hits one is willing to inspect (here 5 % of the list); it weights each ranking's top and ignores its tail.</li>
</ol>

Consensus helps when the rankings are *comparably good* and make *uncorrelated* errors, and hurts when a weak ranking
votes with the same weight as a strong one. Our panel contains one excellent and one poor ranking, so the experiment
below tests that principle rather than demonstrating that consensus works.



In [ ]:
consensus_scores <- function(S, top_frac = 0.05) {
  Z <- scale(as.matrix(S)); R <- apply(as.matrix(S), 2, function(c_) rank(-c_))
  sigma <- top_frac * nrow(S)
  data.frame(cons_z = rowMeans(Z), cons_rank = -rowMeans(R),
             cons_vote = rowSums(R <= top_frac * nrow(S)), cons_ecr = rowSums(exp(-R / sigma)))
}
C_m <- consensus_scores(S_m); all_sc <- cbind(S_m, C_m)
cons_tab <- t(sapply(all_sc, function(s) c(AUC = roc_auc(y_m, s),
                                           BEDROC20 = rie_bedroc(y_m, s, 20)[2],
                                           EF20 = ef(y_m, s, 0.20))))
knitr::kable(cons_tab, digits = 3)


In [ ]:
paired_bootstrap_delta <- function(y, s1, s2, stat = roc_auc, n_boot = 400, seed = 1) {
  set.seed(seed); ai <- which(y == 1); di <- which(y == 0); d <- numeric(n_boot)
  for (b in seq_len(n_boot)) {                   # the SAME replicate scores both rankings
    idx <- c(sample(ai, length(ai), TRUE), sample(di, length(di), TRUE))
    d[b] <- stat(y[idx], s1[idx]) - stat(y[idx], s2[idx])
  }
  c(mean(d), unname(quantile(d, 0.025)), unname(quantile(d, 0.975)), mean(d <= 0))
}
cat("paired bootstrap, delta AUC against the best single ranking (max_tanimoto):\n")
for (c_ in names(C_m)) {
  r <- paired_bootstrap_delta(y_m, C_m[[c_]], S_m$max_tanimoto)
  cat(sprintf("  %-10s %+.4f [%+.4f, %+.4f]  P(delta <= 0) = %.3f\n", c_, r[1], r[2], r[3], r[4]))
}
pair <- rowMeans(scale(as.matrix(S_m[, c("qsar_rf", "desc_dist")])))
r <- paired_bootstrap_delta(y_m, pair, S_m$qsar_rf)
cat(sprintf("\nz-consensus of the two WEAK rankings vs the better of them: AUC %.3f vs %.3f, delta %+.4f [%+.4f, %+.4f]\n",
            roc_auc(y_m, pair), roc_auc(y_m, S_m$qsar_rf), r[1], r[2], r[3]))


In [ ]:
ct <- data.table(ranking = rownames(cons_tab), AUC = cons_tab[, "AUC"])
ct[, kind := ifelse(grepl("^cons", ranking), "consensus", "single")]
ct[, ranking := factor(ranking, levels = ranking[order(AUC)])]
pc1 <- ggplot(ct, aes(AUC, ranking, fill = kind)) + geom_col(width = 0.7) +
  coord_cartesian(xlim = c(0.5, 1)) +
  scale_fill_manual(values = c(single = PAL[["blue"]], consensus = PAL[["orange"]]), name = NULL) +
  labs(title = "Single rankings vs consensus", y = NULL) + theme(legend.position = "bottom")
sp_m <- cor(S_m, method = "spearman")
spdf <- as.data.table(as.table(sp_m)); setnames(spdf, c("a", "b", "rho"))
pc2 <- ggplot(spdf, aes(a, b, fill = rho)) + geom_tile() +
  geom_text(aes(label = sprintf("%.2f", rho)), size = 2.6, colour = "white") +
  scale_fill_gradient(low = PAL[["purple"]], high = PAL[["green"]], limits = c(0, 1)) +
  labs(title = "Spearman correlation between rankings", x = NULL, y = NULL) +
  theme(axis.text.x = element_text(angle = 40, hjust = 1), legend.position = "bottom")
print(pc1 | pc2)



<p class="caption"><b>Figure 13.</b> Left: no consensus scheme beats the best single ranking on this panel. The
z-score consensus loses 0.005 of AUC, rank-by-rank 0.006, ECR 0.005 and rank-by-vote 0.202; in every case the paired
bootstrap interval of the difference excludes zero, so these are small but real losses, not noise. Right: the reason.
The two similarity rankings correlate at Spearman 0.97 — they contribute the same information twice — while the two
property rankings correlate with them at only 0.48-0.52 and are much weaker, so averaging dilutes the good signal with
the bad. Even the consensus of the two weak, comparatively decorrelated rankings fails to beat the better of the pair
(0.955 against 0.968, a loss of 0.013 with interval [-0.019, -0.008]).</p>

One nuance worth noting: rank-by-vote is catastrophic on ROC-AUC (0.792) and yet has the *highest* EF at 20 % of the
whole table (5.00). It is a coarse score with only five distinct values, so almost the entire list is tied; the area
under the curve punishes those ties heavily, while a metric that only looks at the top 20 % is happy with them.
Metric and score granularity interact, which is one more reason to report several metrics.

Consensus is a variance-reduction device, not a free improvement: it pays when components are of similar quality with
different failure modes — typically several docking functions on the same poses — and costs when one component is
clearly better, which is when the temptation to combine everything is strongest. A paired bootstrap of the metric
difference, not two separately reported numbers, is the minimum evidence a consensus claim
needs<sup class="c"><a href="#r4">[4]</a></sup>.

## 14. Score-size correlation and ligand efficiency

Screening scores are often correlated with molecular size: bigger molecules make more contacts, so additive scoring
functions reward them, and the top of a docking list fills with heavy, greasy compounds that are poor starting points
for medicinal chemistry<sup class="c"><a href="#r29">[29]</a></sup>. The remedy is a size-normalised score — **ligand
efficiency**, the free energy of binding per heavy atom,
\(\mathrm{LE} = 1.37 \times \mathrm{pActivity} / N_{\text{heavy}}\) in kcal mol<sup>-1</sup> per
atom<sup class="c"><a href="#r8">[8]</a></sup>. The diagnostic comes first: measure the correlation before correcting
for it.



In [ ]:
ha <- lib$HeavyAtoms[ev_m]
cat("Spearman correlation of each score with heavy-atom count:\n")
for (c_ in names(S_m))
  cat(sprintf("   %-13s rho = %+.3f\n", c_, cor(S_m[[c_]], ha, method = "spearman")))
cat(sprintf("\nranking the library by heavy-atom count ALONE: AUC %.3f (matched decoys), %.3f (random decoys)\n",
            roc_auc(y_m, ha), roc_auc(y_r, lib$HeavyAtoms[ev_r])))
pact <- lib$pActivity[ev_m][y_m == 1]; ha_a <- ha[y_m == 1]
LE <- 1.37 * pact / ha_a
cat(sprintf("actives: pActivity vs heavy atoms rho = %+.3f | ligand efficiency mean %.3f, sd %.3f kcal/mol per atom, %.0f %% below 0.3\n",
            cor(pact, ha_a, method = "spearman"), mean(LE), sd(LE), 100 * mean(LE < 0.3)))
norm <- S_m$max_tanimoto / ha * 30
cat(sprintf("size-normalising the best score: AUC %.3f (unnormalised %.3f)\n",
            roc_auc(y_m, norm), roc_auc(y_m, S_m$max_tanimoto)))


In [ ]:
sz <- data.table(ha = ha, s = S_m$max_tanimoto, y = factor(y_m, labels = c("decoys", "actives")))
pl1 <- ggplot(sz, aes(ha, s, colour = y)) + geom_point(size = 0.4, alpha = 0.25) +
  scale_colour_manual(values = c(decoys = PAL[["blue"]], actives = PAL[["orange"]]), name = NULL) +
  labs(title = "Similarity score vs size (rho = 0.00)", x = "heavy atoms", y = "max Tanimoto") +
  theme(legend.position = c(0.8, 0.8), legend.text = element_text(size = 6))
pl2 <- ggplot(data.table(ha = ha_a, p = pact), aes(ha, p)) +
  geom_point(size = 0.5, alpha = 0.3, colour = PAL[["orange"]]) +
  labs(title = "Potency vs size (actives)", x = "heavy atoms", y = "pActivity")
pl3 <- ggplot(data.table(LE = LE), aes(LE)) + geom_histogram(bins = 40, fill = PAL[["green"]]) +
  geom_vline(xintercept = 0.3, linetype = "dotted") +
  annotate("text", x = 0.305, y = 60, label = "0.3: usual lead criterion", hjust = 0, size = 2.4) +
  labs(title = "Ligand efficiency of the ROCK1 actives",
       x = "ligand efficiency (kcal/mol per heavy atom)", y = "compounds")
print(pl1 | pl2 | pl3)



<p class="caption"><b>Figure 14.</b> Left: the maximum-Tanimoto score is uncorrelated with molecular size
(Spearman -0.004), and ranking the library by heavy-atom count alone gives AUC 0.441 — below chance, because the
actives are slightly smaller than the matched decoys, so the benchmark has no size bias of the usual sign to correct.
Centre: among the actives, potency barely rises with size (rho = +0.043). Right: ligand efficiencies average 0.377
kcal mol<sup>-1</sup> per heavy atom with a tail below the 0.3 value usually required of a lead (14 % of the actives).</p>

A deliberately negative result, and a useful one: size bias belongs to *additive, structure-based* scoring functions,
not to screening in general, and a fingerprint similarity has no reason to grow with atom count. Correcting anyway
costs accuracy — dividing by heavy atoms drops the AUC from 0.993 to 0.987 — by injecting a preference for small
molecules. Diagnose first, correct only if needed; ligand efficiency belongs to hit triage, where it ranks confirmed
actives for chemical tractability<sup class="c"><a href="#r31">[31]</a></sup>. The two property-based scores do show a
mild *negative* size correlation (-0.131 and -0.159), inherited from the same fact: this set of ROCK1 actives is
smaller than its matched decoys.

## 15. Building the list the laboratory will actually buy

The campaign ends with a decision: which fifty compounds. Three criteria compete. **Score** says take the top fifty.
**Diversity** says that fifty analogues of one scaffold test one hypothesis fifty times, so spread the selection over
chemotypes. **Tractability** says remove compounds with known assay-interference substructures (PAINS
filters<sup class="c"><a href="#r33">[33]</a></sup>), implausible properties or impossible chemistry. We implement the
first two at a realistic 1 % prevalence and measure the trade-off; the third is a filter whose cost in hits should be
measured the same way.



In [ ]:
FPe <- FP[ev_m, ]; scaf <- lib$scaffold_smiles[ev_m]; s_best <- S_m$max_tanimoto
select_lists <- function(seed, n_pick = 50, sim_cut = 0.6, pool = 500) {
  set.seed(seed)
  ai <- which(y_m == 1); di <- which(y_m == 0)
  n_a <- round(0.01 * length(di) / 0.99)                  # subsample actives to 1 % prevalence
  idx <- c(sample(ai, n_a), di); yy <- y_m[idx]; ss <- s_best[idx]
  o <- order(-ss, method = "radix"); top <- o[1:n_pick]
  chosen <- o[1]
  for (j in o[2:pool]) {                                   # walk down, keep if dissimilar to all kept
    if (length(chosen) == n_pick) break
    if (max(tani(FPe[idx[j], , drop = FALSE], FPe[idx[chosen], , drop = FALSE])) < sim_cut)
      chosen <- c(chosen, j)
  }
  c(hits_top = sum(yy[top]), hits_div = sum(yy[chosen]),
    scaf_top = length(unique(scaf[idx[top]])), scaf_div = length(unique(scaf[idx[chosen]])),
    n_total = n_a + length(di))
}
res <- as.data.table(t(sapply(0:49, select_lists)))
cat(sprintf("library of %d compounds at 1 %% prevalence; 50 compounds purchased, 50 independent simulations\n",
            res$n_total[1]))
cat(sprintf("top-50 by score      : %.1f +/- %.1f hits, %.1f distinct scaffolds\n",
            mean(res$hits_top), sd(res$hits_top), mean(res$scaf_top)))
cat(sprintf("diverse-50 (T < 0.6) : %.1f +/- %.1f hits, %.1f distinct scaffolds\n",
            mean(res$hits_div), sd(res$hits_div), mean(res$scaf_div)))
cat(sprintf("random 50            : %.2f hits expected\n", 50 * 0.01))


In [ ]:
hl <- melt(res[, .(`top-50 by score` = hits_top, `diverse-50` = hits_div)],
           measure.vars = c("top-50 by score", "diverse-50"), variable.name = "list", value.name = "hits")
ps1 <- ggplot(hl, aes(hits, fill = list)) +
  geom_histogram(binwidth = 1, position = "identity", alpha = 0.7) +
  geom_vline(xintercept = 0.5, linetype = "dotted") +
  annotate("text", x = 1.5, y = 8, label = "random 50", hjust = 0, size = 2.5, angle = 90) +
  scale_fill_manual(values = c(`top-50 by score` = PAL[["blue"]], `diverse-50` = PAL[["orange"]]), name = NULL) +
  labs(title = "Hits bought (1 % prevalence)", x = "confirmed actives among the 50 purchased", y = "simulations") +
  theme(legend.position = "bottom")
tl <- rbind(data.table(list = "top-50 by score", hits = res$hits_top, scaf = res$scaf_top),
            data.table(list = "diverse-50",      hits = res$hits_div, scaf = res$scaf_div))
ps2 <- ggplot(tl, aes(hits, scaf, colour = list)) + geom_point(size = 1.4) +
  scale_colour_manual(values = c(`top-50 by score` = PAL[["blue"]], `diverse-50` = PAL[["orange"]]), name = NULL) +
  labs(title = "The trade-off, one point per simulation",
       x = "hits among the 50 purchased", y = "distinct scaffolds among the 50") +
  theme(legend.position = "bottom")
print(ps1 | ps2)



<p class="caption"><b>Figure 15.</b> Fifty simulated purchase rounds on a 4,196-compound library containing 42 actives.
The fifty best-scoring compounds return 40.5 ± 1.0 confirmed actives spread over 42.7 scaffolds; a Tanimoto &lt; 0.6
dissimilarity filter applied while walking down the list returns 34.2 ± 2.2 actives over 46.6 scaffolds — about six
hits traded for about four chemotypes, both far above the 0.5 hits expected from a random fifty. The two point clouds
in the right panel do not overlap: the trade is systematic, not a matter of luck.</p>

Whether the trade is worth making is a project decision. To confirm a series, buy the top of the list; to find a new
chemotype — the usual reason for screening — the diversity filter is cheap insurance. Report both numbers, expected
hits and expected distinct scaffolds, each with its spread over simulations.

<div class="box bridge"><h4>Domain context — what "buying fifty compounds" means</h4>
<p>A purchase round in this laboratory means ordering milligram quantities of fifty catalogue compounds from a chemical
supplier and running them in a biochemical assay at a single concentration, then measuring a full dose-response curve
for whatever responds. The cost is dominated by the assay, not the compounds, and the turnaround is weeks. That is why
the analytics above matter: the difference between a ranking with hit rate 40 % and one with 4 % is the difference
between one purchase round and ten, and the choice of which fifty to buy is the only place where a data scientist's
work becomes an irreversible, costly commitment.</p></div>

## 16. What a prospective campaign commits to

Every number in this document is retrospective: the labels existed before the ranking was built. A prospective campaign
commits to a protocol in advance and is judged by the assay. The group's checklist, which doubles as the summary of
this unit, is to state in writing before any compound is ordered: the library and its assumed active fraction; the
ranking method with its reference set frozen; the metric and the \(\alpha\) or fraction \(f\) it uses; the number of
compounds that will be bought; the diversity and filtering rules; the expected number of hits computed as
\(k \times \pi \times \mathrm{EF}\); and what result would count as a failure. Published prospective campaigns at very
large scale<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r32">[32]</a></sup> report hit rates of a
few per cent from libraries of hundreds of millions of molecules — orders of magnitude better than random, and still a
large majority of inactive purchases.

Benchmark numbers remain useful for *relative* decisions — which ranking to trust, how many references to collect,
whether consensus earns its pipeline — but they are not predictions. Doing the prevalence arithmetic of section 12 in
front of the collaborators, before the money is spent, is what separates a screening report from a screening
advertisement<sup class="c"><a href="#r7">[7]</a></sup>.



In [ ]:
summary_tab <- data.frame(
  question = c("Does the ranking work?", "How much better than random, early?",
               "Is the difference real?", "Does the decoy set matter?",
               "Is the benchmark honest?", "What will a real campaign get?",
               "Does consensus help?", "Is there a size bias?", "Which 50 to buy?"),
  evidence = c(sprintf("ROC-AUC %.3f (max Tanimoto) vs %.3f (descriptor distance)",
                       roc_auc(y_m, S_m$max_tanimoto), roc_auc(y_m, S_m$desc_dist)),
               sprintf("EF20 %.2f, BEDROC(20) %.3f, hit rate at 50 = %.0f %%",
                       ef(y_m, S_m$max_tanimoto, 0.20), rie_bedroc(y_m, S_m$max_tanimoto, 20)[2],
                       100 * mean(y_m[order(-S_m$max_tanimoto, method = "radix")][1:50])),
               sprintf("stratified bootstrap 95 %% CI [%.3f, %.3f]", ci$auc.lo[1], ci$auc.hi[1]),
               sprintf("AUC gain from matched to random decoys: %+.3f (RF), %+.3f (distance)",
                       decoy_cmp["qsar_rf", "AUC_gain"], decoy_cmp["desc_dist", "AUC_gain"]),
               "properties-only RF: 5-fold AUC 0.978 (matched), 0.999 (random)",
               sprintf("at 0.1 %% prevalence: hit rate in top 100 = %.1f %%, %.1f hits per 50 bought",
                       100 * prev_tab$hit_rate_top100[4], prev_tab$expected_hits_in_50[4]),
               sprintf("best consensus %+.4f AUC vs best single (paired bootstrap)",
                       paired_bootstrap_delta(y_m, C_m$cons_z, S_m$max_tanimoto)[1]),
               sprintf("score vs heavy atoms rho = %+.3f; size-only AUC %.3f",
                       cor(S_m$max_tanimoto, ha, method = "spearman"), roc_auc(y_m, ha)),
               sprintf("top-50: %.1f hits / %.1f scaffolds; diverse-50: %.1f / %.1f",
                       mean(res$hits_top), mean(res$scaf_top), mean(res$hits_div), mean(res$scaf_div))))
knitr::kable(summary_tab)



The table collects the nine decisions this unit can answer with numbers; the tenth — whether the compounds bind — only
an assay can answer.

## 17. Summary

A virtual screen delivers an ordering, and the analytics of that ordering are a self-contained data-science problem.
This unit built four ligand-based rankings of the same 12,018-molecule library from training information only and
measured them from first principles in R. ROC-AUC, implemented as the Mann-Whitney statistic, agrees with `pROC` to six
decimals and places maximum Tanimoto similarity (0.993) and the five-nearest-neighbour score (0.991) above a
property-based random forest (0.968) and a descriptor distance (0.867). Enrichment factors at 1 % and 5 % were unusable
here because three of the four rankings sit on their ceiling 1/prevalence = 6.212; at 20 % they separate the methods
again (4.89, 4.84, 4.44, 3.38). BEDROC and RIE weight the purchasable top at the price of a parameter alpha fixed in
advance; at alpha = 20 the four rankings score 0.997, 0.995, 0.950 and 0.776, and by alpha = 160 the first three are
indistinguishable at 1.000. Stratified bootstrap intervals are narrow here (0.993 [0.989, 0.997] for the best ranking)
and would be several times wider on a typical in-house benchmark with fifty actives instead of 797.

The three honest lessons are larger than any of those numbers. First, the benchmark decides the answer: replacing
property-matched decoys by random ones lifts the random forest from 0.968 to 0.997 and the descriptor distance from
0.867 to 0.961 while barely touching the similarity rankings, and the number of known actives used as references moves
the same algorithm from 0.835 to 0.993, with draws of five references spanning 0.752 to 0.895. Second, a random forest
that sees only twelve bulk properties separates actives from matched decoys at 5-fold AUC 0.978 and from random decoys
at 0.999, so this benchmark cannot support any claim about learnt pharmacology — the artificial-enrichment diagnostic
must be reported with the results. Third, the 16.1 % active fraction of the evaluation set is a fiction: at a realistic
0.1 % the same ranking still has AUC 0.992 and EF<sub>1%</sub> = 96, but the hit rate among the top hundred falls to
3.9 %, about two hits per fifty compounds purchased.

Consensus did not rescue anything — all four aggregation schemes lost to the best single ranking with paired-bootstrap
intervals excluding zero, because two of the four components were weak and the two strong ones were redundant (Spearman
0.97). Size normalisation was equally unnecessary: the similarity score has no size bias to correct (rho = -0.004,
size-only AUC 0.441), and imposing ligand efficiency on it costs 0.006 of AUC. Finally, a diversity filter at Tanimoto
0.6 traded about six hits for about four extra scaffolds in a fifty-compound purchase simulated at 1 % prevalence — the
kind of trade that should be decided by the aim of the campaign and reported as two numbers, not one.

Three habits carry over. Report the composition of any benchmark (prevalence, decoy provenance, reference set) next to
every metric it produced; use a paired test on the same resamples when comparing rankings; and convert retrospective
enrichment into prospective expectation before promising anything to a collaborator. Diversity selection connects to the
clustering of Unit 15 (Butina's leader algorithm<sup class="c"><a href="#r21">[21]</a></sup> solves the same task), and
the activity cliff<sup class="c"><a href="#r22">[22]</a></sup> — a tiny structural change abolishing activity — is why
no similarity ranking is ever more than a prior.

<div class="box bridge"><h4>Bridge</h4>
<p>Within this course: the splits and paired comparisons used here come from <a href="../U12_Validation/index.html">Unit 12</a>,
the random forest from <a href="../U14_Ensembles_Tuning/index.html">Unit 14</a>, the bootstrap from
<a href="../U7_Estimation/index.html">Unit 7</a>, the base-rate reasoning from <a href="../U10_Bayesian/index.html">Unit 10</a>,
the diversity picking and the sparse Tanimoto algebra from <a href="../U15_Clustering/index.html">Unit 15</a>, and the
descriptor matrix from <a href="../U13_Features_Dimensionality/index.html">Unit 13</a>. The next unit,
<a href="../U20_MD_Analytics/index.html">Unit 20</a>, treats the output of a molecular-dynamics run with the same
attitude: correlated samples, honest error bars. The Python edition of this unit, which recomputes the fingerprints
with RDKit instead of reading them, is <a href="../../Python/U19_Virtual_Screening/index.html">here</a>.
In the sister courses: the docking programs whose output this unit analyses are taught in
<a href="../../../BGM324/03_Molecular_Docking/index.html">BGM324 unit 3 (molecular docking)</a>
and the structure-based design workflow in
<a href="../../../BGM324/04_Computer_Aided_Drug_Design/index.html">BGM324 unit 4</a>;
fingerprints and similarity are developed in
<a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">Cheminformatics unit 5</a> and
<a href="../../../Cheminformatics/U11_Similarity_SAR/index.html">unit 11</a>, QSAR validation in
<a href="../../../Cheminformatics/U14_Validation/index.html">Cheminformatics unit 14</a>, and the
machine-learning background in <a href="../../../BGM411/R/index.html">BGM411 (R edition)</a>.</p></div>

## 18. Self-assessment

<details><summary>1. A colleague reports "ROC-AUC 0.95" for a new ligand-based method. Which three pieces of information do you ask for before believing that it is better than maximum Tanimoto similarity?</summary>
<p>The decoy set (provenance and whether it is property-matched), the reference set of known actives (how many, and
whether it is the same one the baseline used), and the evaluation protocol (hold-out or cross-validation, and whether
the baseline was run on exactly the same evaluation molecules). Section 9 showed a 0.16 AUC swing from the size of the
reference set and a 0.14 spread across draws of five references, and section 10 a 0.10 swing from the decoy set.</p></details>

<details><summary>2. Why did three of the four rankings produce exactly the same EF at 1 % and 5 % in section 5?</summary>
<p>Because the enrichment factor is capped at 1/prevalence, and with 16.1 % actives any ranking whose top 1 % and top
5 % contain only actives attains that cap, 6.212. The metric has no resolution left. Either evaluate at a fraction well
above the prevalence, or subsample to a realistic prevalence, or use BEDROC.</p></details>

<details><summary>3. What exactly does the parameter alpha of BEDROC control, and how should it be chosen?</summary>
<p>It sets the exponential decay of the weight given to an active at relative rank x, exp(-alpha x); about 80 % of the
weight lies in the top 4/alpha of the list. Choose it before the analysis from the size of the campaign: if you will
test a fraction f of the library, alpha near 4/f weights the region you can afford. Choosing alpha after seeing the
curves turns the metric into a selection effect.</p></details>

<details><summary>4. A property-only random forest separates your actives from your decoys at AUC 0.98. What does that invalidate, and what does it not?</summary>
<p>It invalidates any claim that performance on this benchmark demonstrates target-specific learning, because a model
with no structural information achieves the same. It does not invalidate relative comparisons between methods
evaluated on the identical decoy set, nor does it mean the ranking will fail prospectively — it means the benchmark
cannot tell you whether it will.</p></details>

<details><summary>5. Your screen has AUC 0.99 and EF at 1 % equal to 96 on a library with one active per thousand compounds. You will buy 50 compounds from the top 1 %. How many hits do you expect, and what is the probability that any given purchased compound is active?</summary>
<p>Hit rate = prevalence x EF = 0.001 x 96 = 0.096, so about 4.8 hits in 50 purchases and a 9.6 % probability per
compound. Against a random draw (0.05 hits) that is a factor of 96, and it still means roughly 45 inactive purchases.
Note that the simulation in section 12 bought the top 100 of the whole list rather than 50 from the top 1 %, and
returned 3.9 %: the arithmetic is only as good as the assumption that the top f is uniform in quality.</p></details>

<details><summary>6. Why must the bootstrap used for a screening metric resample actives and decoys separately?</summary>
<p>Because EF, BEDROC and hit rate all depend on the prevalence of the evaluation set. Resampling the pooled list lets
the number of actives fluctuate, mixing uncertainty about the metric with uncertainty about the benchmark composition,
and inflating the interval. Stratified resampling holds the composition fixed, which is the quantity being studied.</p></details>

<details><summary>7. Under what conditions does consensus scoring help, and how would you demonstrate that it helped?</summary>
<p>When the component rankings are of comparable quality and make uncorrelated errors. Demonstrate it with a paired
bootstrap of the metric difference on the same resamples, reporting the interval of the difference; two separately
quoted AUCs are not evidence. In this unit all four schemes lost to the best single ranking, with intervals excluding
zero.</p></details>

<details><summary>8. A docking campaign returns a top list dominated by molecules of 45-55 heavy atoms. What do you check, and what do you do?</summary>
<p>Check the Spearman correlation between score and heavy-atom count, and the AUC of ranking by size alone. If both
indicate a size effect, re-rank by a size-normalised score (ligand efficiency) or stratify the selection by size bin,
and report both lists. Do not apply the normalisation when the diagnostic is negative, as it was for the similarity
scores here, because it destroys information.</p></details>

<details><summary>9. Why is a diversity-filtered selection usually preferred over the raw top of the list, even though it buys fewer hits?</summary>
<p>Because the purpose of most screens is to find a new chemotype, and fifty analogues of a single scaffold test one
chemical hypothesis fifty times. The simulation in section 15 quantified the price: about six hits for about four
additional scaffolds. Report both numbers and let the project aim decide.</p></details>

<details><summary>10. Which parts of a retrospective evaluation does a bootstrap confidence interval NOT cover?</summary>
<p>Everything that is a design choice rather than a sample: the decoy set, the reference actives, the fingerprint and
similarity measure, the descriptor list, the model's hyperparameters and the metric's alpha. Those are quantified by
re-running the analysis under alternative choices, as in sections 9 to 11, and their effect here was an order of
magnitude larger than the sampling interval.</p></details>

<details><summary>11. You are asked to write the prospective commitment for a fifty-compound campaign. List the items it must contain.</summary>
<p>Library and assumed active fraction; ranking method with a frozen reference set; metric with its alpha or fraction f;
number of compounds to be purchased; diversity and structural-alert filters; the expected number of hits computed as
k x prevalence x EF; and the result that would count as a failure. All of it written before the order is placed.</p></details>

<details><summary>12. The same method gives AUC 0.81 on your benchmark and 0.92 in the paper that proposed it. Name three mundane explanations before concluding that your implementation is wrong.</summary>
<p>A different number or identity of reference actives (Figure 9: 0.75 to 0.90 across draws of five, 0.89 to 0.95
across draws of twenty-five); a different decoy set (Figure 10: up to 0.10 of AUC); and a different evaluation subset or
prevalence. Only after those are matched does a discrepancy point at the code.</p></details>

## Glossary

<dl class="gloss">
<dt>Active</dt><dd>A compound experimentally confirmed to act on the target above a stated potency threshold; here, ROCK1 inhibitors with pActivity at least 6.5.</dd>
<dt>Decoy</dt><dd>A molecule assumed inactive, added to a benchmark so that a ranking can be scored. Property-matched decoys resemble the actives in bulk properties; random decoys do not.</dd>
<dt>Prevalence (active fraction)</dt><dd>Proportion of actives in the evaluated set. Determines the ceiling of the enrichment factor and the hit rate of any purchase.</dd>
<dt>ROC curve / ROC-AUC</dt><dd>True-positive rate against false-positive rate along the ranked list; its area is the probability that a random active outranks a random decoy.</dd>
<dt>Enrichment factor (EF)</dt><dd>Hit rate in the top fraction f divided by the hit rate of the whole library; bounded above by 1/prevalence.</dd>
<dt>BEDROC / RIE</dt><dd>Early-recognition metrics that weight an active at relative rank x by exp(-alpha x); BEDROC is RIE rescaled to [0, 1] at the given prevalence.</dd>
<dt>Hit rate at k</dt><dd>Fraction of actives among the first k compounds of the ranking - the quantity a purchase decision is made on.</dd>
<dt>Artificial enrichment</dt><dd>Separability of actives from decoys that comes from benchmark construction (property differences) rather than from target pharmacology.</dd>
<dt>Analogue bias</dt><dd>Over-representation of close analogues among the actives, which inflates similarity-based metrics because one scaffold is found many times.</dd>
<dt>Tanimoto coefficient</dt><dd>Shared bits divided by the union of set bits of two fingerprints; the standard similarity for Morgan fingerprints, written here as sparse-matrix algebra.</dd>
<dt>Consensus / rank aggregation</dt><dd>Combining several rankings by averaging standardised scores, averaging ranks, counting votes or summing exponentially weighted ranks (ECR).</dd>
<dt>Ligand efficiency</dt><dd>1.37 x pActivity divided by the heavy-atom count: potency per atom, used to compare molecules of different size.</dd>
<dt>Scaffold (Bemis-Murcko)</dt><dd>The ring systems and their linkers after side chains are removed; the usual unit of chemical-series bookkeeping.</dd>
<dt>Prospective validation</dt><dd>Buying and testing compounds selected by a frozen protocol; the only evaluation immune to benchmark artefacts.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Truchon JF, Bayly CI. Evaluating virtual screening methods: good and bad metrics for the "early recognition" problem. <em>J Chem Inf Model</em>. 2007;47(2):488-508. <a href="https://doi.org/10.1021/ci600426e">doi:10.1021/ci600426e</a></li>
<li id="r2">Mysinger MM, Carchia M, Irwin JJ, Shoichet BK. Directory of Useful Decoys, Enhanced (DUD-E): better ligands and decoys for better benchmarking. <em>J Med Chem</em>. 2012;55(14):6582-6594. <a href="https://doi.org/10.1021/jm300687e">doi:10.1021/jm300687e</a></li>
<li id="r3">Sieg J, Flachsenberg F, Rarey M. In need of bias control: evaluating chemical data for machine learning in structure-based virtual screening. <em>J Chem Inf Model</em>. 2019;59(3):947-961. <a href="https://doi.org/10.1021/acs.jcim.8b00712">doi:10.1021/acs.jcim.8b00712</a></li>
<li id="r4">Clark RD, Webster-Clark DJ. Managing bias in ROC curves. <em>J Comput Aided Mol Des</em>. 2008;22(3-4):141-146. <a href="https://doi.org/10.1007/s10822-008-9181-z">doi:10.1007/s10822-008-9181-z</a></li>
<li id="r5">Palacio-Rodriguez K, Lans I, Cavasotto CN, Cossio P. Exponential consensus ranking improves the outcome in docking and receptor ensemble docking. <em>Sci Rep</em>. 2019;9:5142. <a href="https://doi.org/10.1038/s41598-019-41594-3">doi:10.1038/s41598-019-41594-3</a></li>
<li id="r6">Willett P. Similarity-based virtual screening using 2D fingerprints. <em>Drug Discov Today</em>. 2006;11(23-24):1046-1053. <a href="https://doi.org/10.1016/j.drudis.2006.10.005">doi:10.1016/j.drudis.2006.10.005</a></li>
<li id="r7">Ripphausen P, Nisius B, Bajorath J. State-of-the-art in ligand-based virtual screening. <em>Drug Discov Today</em>. 2011;16(9-10):372-376. <a href="https://doi.org/10.1016/j.drudis.2011.02.011">doi:10.1016/j.drudis.2011.02.011</a></li>
<li id="r8">Hopkins AL, Keseru GM, Leeson PD, Rees DC, Reynolds CH. The role of ligand efficiency metrics in drug discovery. <em>Nat Rev Drug Discov</em>. 2014;13(2):105-121. <a href="https://doi.org/10.1038/nrd4163">doi:10.1038/nrd4163</a></li>
<li id="r9">Lyu J, Wang S, Balius TE, et al. Ultra-large library docking for discovering new chemotypes. <em>Nature</em>. 2019;566:224-229. <a href="https://doi.org/10.1038/s41586-019-0917-9">doi:10.1038/s41586-019-0917-9</a></li>
<li id="r10">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r11">Bender A, Glen RC. Molecular similarity: a key technique in molecular informatics. <em>Org Biomol Chem</em>. 2004;2(22):3204-3218. <a href="https://doi.org/10.1039/b409813g">doi:10.1039/b409813g</a></li>
<li id="r12">Jain AN, Nicholls A. Recommendations for evaluation of computational methods. <em>J Comput Aided Mol Des</em>. 2008;22(3-4):133-139. <a href="https://doi.org/10.1007/s10822-008-9196-5">doi:10.1007/s10822-008-9196-5</a></li>
<li id="r13">Nicholls A. What do we know and when do we know it? <em>J Comput Aided Mol Des</em>. 2008;22(3-4):239-255. <a href="https://doi.org/10.1007/s10822-008-9170-2">doi:10.1007/s10822-008-9170-2</a></li>
<li id="r14">Hawkins PCD, Warren GL, Skillman AG, Nicholls A. How to do an evaluation: pitfalls and traps. <em>J Comput Aided Mol Des</em>. 2008;22(3-4):179-190. <a href="https://doi.org/10.1007/s10822-008-9189-4">doi:10.1007/s10822-008-9189-4</a></li>
<li id="r15">Reau M, Langenfeld F, Zagury JF, Lagarde N, Montes M. Decoys selection in benchmarking datasets: overview and perspectives. <em>Front Pharmacol</em>. 2018;9:11. <a href="https://doi.org/10.3389/fphar.2018.00011">doi:10.3389/fphar.2018.00011</a></li>
<li id="r16">Chen L, Cruz A, Ramsey S, et al. Hidden bias in the DUD-E dataset leads to misleading performance of deep learning in structure-based virtual screening. <em>PLoS One</em>. 2019;14(8):e0220113. <a href="https://doi.org/10.1371/journal.pone.0220113">doi:10.1371/journal.pone.0220113</a></li>
<li id="r17">Wallach I, Heifets A. Most ligand-based classification benchmarks reward memorization rather than generalization. <em>J Chem Inf Model</em>. 2018;58(5):916-932. <a href="https://doi.org/10.1021/acs.jcim.7b00403">doi:10.1021/acs.jcim.7b00403</a></li>
<li id="r18">Tran-Nguyen VK, Jacquemard C, Rognan D. LIT-PCBA: an unbiased data set for machine learning and virtual screening. <em>J Chem Inf Model</em>. 2020;60(9):4263-4273. <a href="https://doi.org/10.1021/acs.jcim.0c00155">doi:10.1021/acs.jcim.0c00155</a></li>
<li id="r19">Rohrer SG, Baumann K. Maximum unbiased validation (MUV) data sets for virtual screening based on PubChem bioactivity data. <em>J Chem Inf Model</em>. 2009;49(2):169-184. <a href="https://doi.org/10.1021/ci8002649">doi:10.1021/ci8002649</a></li>
<li id="r20">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r21">Butina D. Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity: a fast and automated way to cluster small and large data sets. <em>J Chem Inf Comput Sci</em>. 1999;39(4):747-750. <a href="https://doi.org/10.1021/ci9803381">doi:10.1021/ci9803381</a></li>
<li id="r22">Maggiora GM. On outliers and activity cliffs - why QSAR often disappoints. <em>J Chem Inf Model</em>. 2006;46(4):1535. <a href="https://doi.org/10.1021/ci060117s">doi:10.1021/ci060117s</a></li>
<li id="r23">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r24">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r25">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
<li id="r26">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r27">Hanley JA, McNeil BJ. The meaning and use of the area under a receiver operating characteristic (ROC) curve. <em>Radiology</em>. 1982;143(1):29-36. <a href="https://doi.org/10.1148/radiology.143.1.7063747">doi:10.1148/radiology.143.1.7063747</a></li>
<li id="r28">Efron B. Bootstrap methods: another look at the jackknife. <em>Ann Stat</em>. 1979;7(1):1-26. <a href="https://doi.org/10.1214/aos/1176344552">doi:10.1214/aos/1176344552</a></li>
<li id="r29">Scior T, Bender A, Tresadern G, et al. Recognizing pitfalls in virtual screening: a critical review. <em>J Chem Inf Model</em>. 2012;52(4):867-881. <a href="https://doi.org/10.1021/ci200528d">doi:10.1021/ci200528d</a></li>
<li id="r30">Macarron R, Banks MN, Bojanic D, et al. Impact of high-throughput screening in biomedical research. <em>Nat Rev Drug Discov</em>. 2011;10(3):188-195. <a href="https://doi.org/10.1038/nrd3368">doi:10.1038/nrd3368</a></li>
<li id="r31">Kuntz ID, Chen K, Sharp KA, Kollman PA. The maximal affinity of ligands. <em>Proc Natl Acad Sci USA</em>. 1999;96(18):9997-10002. <a href="https://doi.org/10.1073/pnas.96.18.9997">doi:10.1073/pnas.96.18.9997</a></li>
<li id="r32">Gorgulla C, Boeszoermenyi A, Wang ZF, et al. An open-source drug discovery platform enables ultra-large virtual screens. <em>Nature</em>. 2020;580:663-668. <a href="https://doi.org/10.1038/s41586-020-2117-z">doi:10.1038/s41586-020-2117-z</a></li>
<li id="r33">Baell JB, Holloway GA. New substructure filters for removal of pan assay interference compounds (PAINS) from screening libraries. <em>J Med Chem</em>. 2010;53(7):2719-2740. <a href="https://doi.org/10.1021/jm901137j">doi:10.1021/jm901137j</a></li>
<li id="r34">McGaughey GB, Sheridan RP, Bayly CI, et al. Comparison of topological, shape, and docking methods in virtual screening. <em>J Chem Inf Model</em>. 2007;47(4):1504-1519. <a href="https://doi.org/10.1021/ci700052x">doi:10.1021/ci700052x</a></li>
<li id="r35">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. - Chapter 4 for ROC curves and classification metrics, with R laboratories.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. - Sections 7.10 and 9.2 for resampling and trees.</li>
<li>Leach AR, Gillet VJ. <em>An Introduction to Chemoinformatics</em>. Rev. ed. Springer; 2007. - Similarity searching, virtual screening and diversity selection.</li>
<li>Wickham H, Cetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023 (free online). - The data-handling idioms used throughout this edition.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>TeachOpenCADD talktorials on ligand-based screening and enrichment: <a href="https://projects.volkamerlab.org/teachopencadd/">projects.volkamerlab.org/teachopencadd</a>.</li>
<li>Practical Cheminformatics (P. Walters), posts on evaluating virtual screening and on the statistics of hit lists: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
<li><code>ChemmineR</code>, the Bioconductor cheminformatics toolkit for R (fingerprints and similarity from SDF): <a href="https://bioconductor.org/packages/ChemmineR/">bioconductor.org/packages/ChemmineR</a>.</li>
<li>CRAN Task View "Chemometrics and Computational Physics": <a href="https://cran.r-project.org/view=ChemPhys">cran.r-project.org/view=ChemPhys</a>.</li>
<li>DUD-E and LIT-PCBA benchmark sites: <a href="https://dude.docking.org/">dude.docking.org</a>, <a href="https://drugdesign.unistra.fr/LIT-PCBA/">drugdesign.unistra.fr/LIT-PCBA</a>.</li>
</ul>



